# PRAM: closed testing run exactly as published (notebook 02)

**What this notebook does**: it runs the closed-testing procedure of Vergouwe et al. (2017) **exactly as published** next to the variant used in the paper (`update_hierarchy` of notebook 01) and puts the two side by side. The variant differs from the published procedure in two places: the order of the tests and the prediction used when the revision is chosen.

**Published procedure** (Vergouwe 2017, page 4531, steps B to D): (1) if revision versus no update (p + 1 degrees of freedom) is not significant, **no update**. (2) Otherwise, if revision versus intercept update (p degrees of freedom) is not significant, **intercept update**. (3) Otherwise, if revision versus recalibration (p - 1 degrees of freedom) is not significant, **recalibration**, and if it is significant, **revision**. The revision is an unpenalized maximum-likelihood re-estimation; shrinkage is outside the procedure (page 4536).

**Steps of the notebook** (it must be opened in the working folder in which notebook 01 finished)

1. It only **reads** the grid and hospital-level results in `out_pram/resume/` made by notebook 01 (values of the variant, retrieval and the frozen models).
2. With the same split, the same draw key (`m23`) and the same banks, it fits **only the published procedure**. This is the only new unit. No section below can write to an existing resume file (an attempt stops the run before computing).
3. Paired with the variant cell by cell, it gives the step distribution, AUROC, ECE, net benefit and the comparison with retrieval (PRAM). All new file names start with `CTPUB_`.

**How to run**: open this notebook in the working folder in which notebook 01 was run (where `out_pram` is) and run the cells from top to bottom. Do not change the settings. At the end, `out_pram_ctpub.zip` is written to the current folder.

**If the folder differs**, the write-lock and state-check cell stops before computing and prints the reason (it stops if the grid results of notebook 01 are absent or insufficient). No new computation is started.


In [ ]:
# ================================================================
# PRAM Cell 1. Settings and input check (all constants live in this cell only)
# ================================================================
import os, sys, io, gc, json, time, math, zipfile, hashlib, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from scipy.optimize import minimize, minimize_scalar

import sklearn
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, brier_score_loss, average_precision_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.isotonic import IsotonicRegression
from sklearn.exceptions import ConvergenceWarning
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


def _ov(name, default):
    """Used only when a test overrides values. Defaults apply in a normal run."""
    return globals().get(name + "_OVERRIDE", default)


# ---- run profile ---------------------------------------------------
#   "full"  : for the paper (default). The repetition counts are set in one place below.
#   "check" : pipeline check. Only verifies that the run completes on real data. Results are not used in the paper.
RUN_PROFILE = _ov("RUN_PROFILE", "full")
_PROFILES = {
    "full":  dict(N_DRAWS_MAIN=10, N_DRAWS_TAIL=100, BANK_SIZES=[500, 1000, 2000, 5000, "full"],
                  TAIL_BANKS=[500, 1000, 2000, 5000], LIGHT_BANKS=[1000, "full"], SITE_LIMIT=None,
                  TABPFN_BANKS=[500, 1000, 5000, "full"], COLDSTART_BANKS=[0, 100, 300, 500, 1000, 2000, 5000, "full"]),
    "check": dict(N_DRAWS_MAIN=2, N_DRAWS_TAIL=10, BANK_SIZES=[500, "full"],
                  TAIL_BANKS=[500], LIGHT_BANKS=["full"], SITE_LIMIT=8, TABPFN_BANKS=[500, "full"], COLDSTART_BANKS=[0, 500, "full"]),
}
_P = _PROFILES[RUN_PROFILE]

# ---- shared settings (seed, bootstrap repetitions, input folders) are defined once in code/pram_config.py ----
_cfg_dir = next((p / "code" for p in [Path.cwd(), *Path.cwd().parents] if (p / "code" / "pram_config.py").is_file()), None)
if _cfg_dir is None:
    raise FileNotFoundError("code/pram_config.py not found: open this notebook inside the repository checkout")
sys.path.insert(0, str(_cfg_dir))
from pram_config import SEED, N_BOOT_WITHIN, THRESH_RANGE, DATA_ROOT, MIMIC4_ROOT, EICU_ROOT  # noqa: E402

# ---- reproducibility and repetition counts (one value for the whole analysis) --------------------------
ALPHA_GRID = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]   # mixing weight grid
K_NEIGHBORS = 50                            # default of retrieval_R1_cosine (Methods)
N_FOLDS = 5                                 # number of mock-batch folds
N_INNER = 3                                 # number of inner cross-validation folds within a bank (inner_cv_bank)
N_SITE_SPLITS = 3                           # number of folds in the hospital-level analysis
N_DRAWS_MAIN = _ov("N_DRAWS_MAIN", _P["N_DRAWS_MAIN"])    # number of draws per bank size
N_DRAWS_TAIL = _ov("N_DRAWS_TAIL", _P["N_DRAWS_TAIL"])    # tail risk: bank draws per cell = 5 folds x 20 draws per fold (100, as in Supplementary S7 Panel D)
BANK_SIZES = _ov("BANK_SIZES", _P["BANK_SIZES"])          # bank sizes of the strategy grid
TAIL_BANKS = _ov("TAIL_BANKS", _P["TAIL_BANKS"])          # the four bank sizes of Supplementary S7 Panel D
TABPFN_BANKS = _ov("TABPFN_BANKS", _P["TABPFN_BANKS"])
LIGHT_BANKS = _ov("LIGHT_BANKS", _P["LIGHT_BANKS"])       # bank sizes of the variants without temperature only and without INR only
SITE_LIMIT = _ov("SITE_LIMIT", _P["SITE_LIMIT"])          # None means all eligible hospitals
DEV_CB_ITER = _ov("DEV_CB_ITER", 500)       # frozen CatBoost iterations
LOCAL_CB_ITER = _ov("LOCAL_CB_ITER", 300)   # local CatBoost iterations
LAM_GRID = np.logspace(-4, 4, 17)           # anchored ridge regularization grid
ANCHORED_MAX_BANK = None                    # largest bank size at which the anchored ridge (model revision step) is fitted. None also fits the full bank
REVISION_RIDGE = 1e-5                       # tiny regularization for numerical stability of the model-revision fit (mean-loss units)
CLOSED_TEST_ALPHA = 0.05                    # closed-testing significance level
N_EVENT_MIN = 2                             # minimum number of events and non-events in a bank (below this the update fit is skipped)
N_TIME_SPLITS = 2                           # A1 temporal validation: number of folds that split the test period by outcome stratum (test fold and same-period bank alternate)

# ---- population and outcome definitions ----------------------
DROP_COLS = ["crp_min", "crp_max", "crp_last", "albumin_last", "albumin_max", "albumin_min",
             "bmi", "height_cm_min", "height_cm_last", "height_cm_max"]
OUTCOME_MAP = {"AKI_168h": "aki_168h", "mortality_168h": "death_168h"}
OUTCOMES = ["AKI_168h", "mortality_168h"]
TARGETS = ["MIMIC-IV", "MIMIC-III", "eICU"]
MISS_CUT_PCT = 30                           # variables whose maximum missingness over the three databases exceeds this value are dropped from the 33-column set
EXPECTED_SPARSE_9 = ["inr_last", "inr_max", "inr_min", "temp_c_last", "temp_c_max", "temp_c_min",
                     "weight_kg_last", "weight_kg_max", "weight_kg_min"]      # the nine variables expected to be dropped

# ---- hospital level and network ---------------------------
MIN_SITE_N = _ov("MIN_SITE_N", 300)
MIN_SITE_EVENTS = _ov("MIN_SITE_EVENTS", 15)    # minimum number of events and of non-events each
MAX_PARTNERS = 20
TEST_FRAC_NET = 0.30
NET_ALPHA_FIXED = 0.30                      # fixed mixing weight of an unlabeled hospital
NET_ALPHA_GRID = [round(a, 2) for a in np.arange(0.0, 1.0001, 0.05)]   # 0 to 1.0 in steps of 0.05
PARTNER_PREV_FACTORS = _ov("PARTNER_PREV_FACTORS", [1.0])                # partner bank prevalence multipliers. With 1.0 only, the whole partner cohort is used. Changing to [0.5, 1.0, 2.0] turns on the partner prevalence scenarios of A7

# ---- net benefit thresholds (A3) --------------------------------------------------
THRESH_GRID = np.round(np.arange(0.01, 0.5001, 0.01), 2)      # full 1 to 50% grid, stored in all tables
THRESH_MS = 0.20                            # main threshold of the main analysis
#   THRESH_RANGE (outcome-specific threshold ranges) is defined in code/pram_config.py; the summary changes when the ranges are edited.
#   the full-grid (1 to 50%) net benefit is written to csv, so changing the range needs no rerun.
T_SITE = {oc: round(float(np.mean(r)), 2) for oc, r in THRESH_RANGE.items()}   # midpoints of the outcome-specific ranges (representative thresholds of A2 and A11)
RANGE_ORIGINAL = (0.02, 0.30)               # range 2 to 30%

# ---- A2 tail risk and non-inferiority margin (post hoc definitions, labeled post hoc in the summary) ---------
NI_MARGINS_AUROC = [0.0, 0.005, 0.01, 0.02]
NI_MARGIN_AUROC = 0.01
NI_MARGINS_NB = [0.0, 0.0005, 0.001, 0.002]
NI_MARGIN_NB = 0.001                        # net benefit 0.001 = one net true positive per 1,000 patients
CP_LEVEL = 0.95                             # Clopper-Pearson interval level
SAFE_UPPER_BOUND = 0.05                     # "safest minimum bank size" = first size at which the upper bound of the probability of losing is at or below this value

A2_TARGETS_F33 = ["eICU"]                   # targets for which the 33-column tail is computed anew (eICU)
A2_RECOMPUTE_F42 = _ov("A2_RECOMPUTE_F42", False)                    # if False and the reference 42-column tail file (`a54_alpha_stability.jsonl`) exists in REFERENCE_RESUME_DIR, the 42-column tail is read from it (AUROC gain only, no model fit)
REFERENCE_RESUME_DIR = Path("reference") / "resume"                            # folder of stored reference result files (optional)
A54_PATH = REFERENCE_RESUME_DIR / "a54_alpha_stability.jsonl"
A29_PATH = REFERENCE_RESUME_DIR / "a29_real_sites.jsonl"
A51_PATH = REFERENCE_RESUME_DIR / "a51_hospital_thresholds.jsonl"
A59_PATH = REFERENCE_RESUME_DIR / "a59_hospital_diag.jsonl"
REPRO_UNITS = _ov("REPRO_UNITS", [("AKI_168h", "eICU", 1000), ("mortality_168h", "eICU", "full"),
                                  ("AKI_168h", "MIMIC-III", 500), ("mortality_168h", "MIMIC-IV", 2000)])   # units compared with the reference values before the 42-column grid
REPRO_TOL = 0.00005                         # the reference values have four decimals, so a rounding tolerance is allowed

# ---- rescoring: per-draw paired DeLong (Table 2 q column, Supplementary S7 Panels A and B, development-bank starting table p column) -------------
#   Only the two sections below (rescoring and starting bank) compute anew. The other sections skip units already finished in resume.
RUN_RESCORE = _ov("RUN_RESCORE", True)               # turns on the rescoring section (pooled DeLong, 42 columns) and its pre-check
RESCORE_COLDSTART = _ov("RESCORE_COLDSTART", True)   # p column of the development-bank starting table. Turning it off skips the computation of that column
RESCORE_ONLY_GUARD = _ov("RESCORE_ONLY_GUARD", False)  # if True, stops before computing when a unit outside the rescoring would run anew (guards against rerunning 12 to 16 hours in a folder that already holds finished results). Off by default so that the notebook runs from scratch
RESCORE_NAMES = ("rescore_F42", "coldstart_F42")      # only the resume file with this name is written anew
COLDSTART_BANKS = _ov("COLDSTART_BANKS", _P["COLDSTART_BANKS"])   # 0 = source bank only, number = source bank + local bank
COLDSTART_SRC_OOF_SPLITS = 3                # number of cross-validation folds for the source-bank base prediction of _source_bank
SIG_LEVEL = 0.05                            # significance level of the per-draw test (criterion of Supplementary S7 Panel B)

# ---- A1 temporal validation ------------------------------------------------------
A1_POLICIES = _ov("A1_POLICIES", ("P", "S"))                    # P past bank, S same-period bank. Changing to ("P", "S", "R") also gives a random (periods mixed) bank R
HOSP_MIN_N_PER_YEAR = _ov("HOSP_MIN_N_PER_YEAR", 300)    # hospital-level temporal analysis: minimum number of ICU stays in each of the two years
HOSP_MIN_EVENTS_PER_YEAR = _ov("HOSP_MIN_EVENTS_PER_YEAR", 15)
HOSP_TIME_MIN_K = 10                        # if fewer hospitals qualify, only descriptive statistics instead of meta-analysis
RUN_A1C = _ov("RUN_A1C", False)                             # A1c: refit the development model on the earlier MIMIC-IV period (turned off)
YEAR_RULES = ("strict", "majority")         # MIMIC-IV year-period assignment: only when entirely inside a period / period covering the majority
RUN_A1D = _ov("RUN_A1D", False)                             # A1d: hospital-level temporal validation (turned off; the number of hospitals above the criterion in both years is unknown)

# ---- A9 TabPFN ----------------------------------------------------------
TABPFN_N_ESTIMATORS = _ov("TABPFN_N_ESTIMATORS", 4)
TABPFN_CTX_GPU = _ov("TABPFN_CTX_GPU", 10000)       # development subsample size (context limit of v2: 10,000 rows)
TABPFN_CTX_CPU = _ov("TABPFN_CTX_CPU", 2000)
TABPFN_SUBSETS_GPU = _ov("TABPFN_SUBSETS_GPU", 3)   # number of subsample ensemble members
TABPFN_SUBSETS_CPU = _ov("TABPFN_SUBSETS_CPU", 1)
TABPFN_CPU_TARGET_CAP = _ov("TABPFN_CPU_TARGET_CAP", 6000)   # maximum number of rows per target cohort on CPU
TABPFN_MAX_MINUTES = _ov("TABPFN_MAX_MINUTES", 240)          # if the expected time exceeds this, the sample is reduced
TABPFN_BATCH = _ov("TABPFN_BATCH", 128)   # rows per prediction batch. On an 8 GB GPU a batch of 10,000 context rows x 5,000 test rows requires 72 GiB and runs out of memory. Start at 128 and halve and retry on out-of-memory (tabpfn_predict)
TABPFN_TARGETS = ["eICU", "MIMIC-III"]              # eICU and MIMIC-III external validation

# ---- A11 strategy selection rule -------------------------------------------------
SEL_CANDIDATES = ["b1_base", "b5_base", "platt_b1", "recal_iso", "pram_b1", "pram_b5", "local_lr"]
#   pooled_lr needs the development record and cannot be chosen by a hospital, and local_cb and anchored ridge are left out of the candidates because their inner cross-validation within a bank is costly.

# ---- sections to run (each can be switched on or off; off removes both its computation and its report) -------------
RUN_MATRIX = _ov("RUN_MATRIX", True)       # strategy grid (main analysis, the 33-column set of A8, A4, A5 and A11 are derived from it)
RUN_A1 = _ov("RUN_A1", True)           # temporal validation
RUN_A2 = _ov("RUN_A2", True)           # tail risk
RUN_HOSPITAL = _ov("RUN_HOSPITAL", True)     # hospital-level computation (the hospital parts of A3, A6 and A8 are derived from it)
RUN_A3 = _ov("RUN_A3", True)           # decision-curve threshold report (on top of the hospital-level results)
RUN_A4 = _ov("RUN_A4", True)           # update hierarchy and closed testing (hier and anch groups from the grid and the hospital level)
RUN_A5 = _ov("RUN_A5", True)           # retrieval + Platt combination order
RUN_A6 = _ov("RUN_A6", True)           # hospital-level meta-analysis (tau, prediction interval, event counts)
RUN_A7 = _ov("RUN_A7", True)           # network of unlabeled hospitals and partner prevalence (including the network part of A8). The 33-column network is also derived from it
RUN_A9 = _ov("RUN_A9", True)           # TabPFN
RUN_REAGG = _ov("RUN_REAGG", True)      # if the reference hospital-level files (`a29_real_sites`, `a51_hospital_thresholds`, `a59_hospital_diag`) exist in REFERENCE_RESUME_DIR, recount them and compare with the hospital computation of this notebook (skipped if absent)
RUN_A11 = _ov("RUN_A11", False)         # strategy selection rule within a bank (turned off)

# ---- strategy group ----------------------------------------------------------
GROUPS_FULL = (("frozen", "pram", "recal", "local", "localcb") + (("anch", "hier") if RUN_A4 else ())
               + (("combo",) if RUN_A5 else ()) + (("sel",) if RUN_A11 else ()))
GROUPS_LIGHT = ("frozen", "pram", "recal")
GROUPS_HOSP = ("frozen", "pram", "recal", "local", "localcb") + (("anch", "hier") if RUN_A4 else ()) + (("combo",) if RUN_A5 else ())
M_EIGHT = ["b1_base", "b5_base", "recal_iso", "pram_b1", "pram_b5", "local_lr", "pooled_lr", "local_cb"]   # the eight strategies of the manuscript
FS_SPEC = {
    "F42":        dict(drop_rule=None, drop_prefix=(), targets=TARGETS, banks=BANK_SIZES,
                       groups=GROUPS_FULL, hospital=True, network=True),
    "F33":        dict(drop_rule="miss", drop_prefix=(), targets=TARGETS, banks=BANK_SIZES,
                       groups=GROUPS_FULL, hospital=True, network=True),
    "F42_noTemp": dict(drop_rule=None, drop_prefix=("temp_c_",), targets=["eICU"], banks=LIGHT_BANKS,
                       groups=GROUPS_LIGHT, hospital=False, network=False),
    "F42_noINR":  dict(drop_rule=None, drop_prefix=("inr_",), targets=["eICU"], banks=LIGHT_BANKS,
                       groups=GROUPS_LIGHT, hospital=False, network=False),
}
FS_RUN = _ov("FS_RUN", ["F42", "F33"])      # "F42_noTemp" and "F42_noINR" (variants without temperature only or INR only) run when included

# ---- output --------------------------------------------------------------
OUT_NAME = "out_pram" + ("" if RUN_PROFILE == "full" else "_CHECK")
OUT_DIR = Path(OUT_NAME)
CSV_DIR = OUT_DIR / "github_csv"
RESUME_DIR = OUT_DIR / "resume"
for _d in (OUT_DIR, CSV_DIR, RESUME_DIR):
    _d.mkdir(parents=True, exist_ok=True)
ZIP_NAME = OUT_NAME + ".zip"

# ---- candidate paths of the temporal and hospital raw data --------------------------
#   candidates for the eICU patient table are the list below plus the .gz variants.
#   MIMIC-IV candidates are listed below. If not found, write the actual path in EXTRA_RAW_PATHS below.
EXTRA_RAW_PATHS = {"eicu_patient": [EICU_ROOT / "patient.csv", EICU_ROOT / "patient.csv.gz"], "mimic4_patients": [], "mimic4_icustays": []}
#   MIMIC-IV folder (patients and icustays are expected under hosp and icu in this folder).
#   Below, hosp/patients.csv(.gz) and icu/icustays.csv(.gz) are searched. If not found, the paths searched are printed in the input check table and only A1b is skipped.
_E = ["./eICU/patient.csv", "eICU/patient.csv", "./eicu/patient.csv",
      DATA_ROOT / "eICU" / "patient.csv", DATA_ROOT / "eicu" / "patient.csv", DATA_ROOT / "eicu" / "hospital_map.csv"]
EICU_PATIENT_CANDIDATES = [Path(p) for p in _E + [str(p) + ".gz" for p in _E[:5]]
                           + EXTRA_RAW_PATHS["eicu_patient"]]
_M4DIRS = [MIMIC4_ROOT, DATA_ROOT / "mimic4", "."]
MIMIC4_PATIENTS_CANDIDATES = [Path(d) / s for d in _M4DIRS for s in ("hosp/patients.csv", "hosp/patients.csv.gz", "patients.csv")] \
    + [Path(p) for p in EXTRA_RAW_PATHS["mimic4_patients"]]
MIMIC4_ICUSTAYS_CANDIDATES = [Path(d) / s for d in _M4DIRS for s in ("icu/icustays.csv", "icu/icustays.csv.gz", "icustays.csv")] \
    + [Path(p) for p in EXTRA_RAW_PATHS["mimic4_icustays"]]


def find_first(cands):
    """the first existing path among the candidates, and the list of absolute paths searched."""
    tried = []
    for p in cands:
        try:
            ap = Path(p).resolve()
        except OSError:
            ap = Path(p)
        tried.append(str(ap))
        try:
            if Path(p).is_file():
                return Path(p), tried
        except OSError:
            continue
    return None, tried


def header_cols(path):
    try:
        return [str(c) for c in pd.read_csv(path, nrows=3).columns]
    except Exception as exc:  # noqa: BLE001
        print("  [read failed]", path, type(exc).__name__, str(exc)[:80])
        return []


# ---- input check table ---------------------------------------------------------
MAIN_FILES = {
    "MIMIC-IV features": DATA_ROOT / "mimic4" / "X_folded.parquet",
    "MIMIC-IV outcomes": DATA_ROOT / "mimic4" / "binary_outcomes_mimic4.csv",
    "MIMIC-III features": DATA_ROOT / "mimic3" / "X_external.parquet",
    "MIMIC-III outcomes": DATA_ROOT / "mimic3" / "binary_outcomes_mimic3.csv",
    "eICU features": DATA_ROOT / "eicu" / "X_external.parquet",
    "eICU outcomes": DATA_ROOT / "eicu" / "binary_outcomes_eicu.csv",
}
INPUT_ROWS = []
for _k, _p in MAIN_FILES.items():
    INPUT_ROWS.append((_k, str(Path(_p).resolve()), "present" if Path(_p).is_file() else "absent"))
DATA_OK = all(r[2] == "present" for r in INPUT_ROWS)

EICU_PATIENT_PATH, _tried_e = find_first(EICU_PATIENT_CANDIDATES)
EICU_COLS = header_cols(EICU_PATIENT_PATH) if EICU_PATIENT_PATH else []
_lowE = {c.lower(): c for c in EICU_COLS}
EICU_COL_STAY = _lowE.get("patientunitstayid")
EICU_COL_HOSP = _lowE.get("hospitalid")
EICU_COL_YEAR = _lowE.get("hospitaldischargeyear")
HAVE_HOSPITALID = bool(EICU_COL_STAY and EICU_COL_HOSP)
HAVE_EICU_YEAR = bool(EICU_COL_STAY and EICU_COL_YEAR)
INPUT_ROWS.append(("eICU patient table (hospital id)", str(EICU_PATIENT_PATH) if EICU_PATIENT_PATH else "not found",
                   "column present" if HAVE_HOSPITALID else "column absent"))
INPUT_ROWS.append(("eICU patient table (hospitaldischargeyear)", str(EICU_PATIENT_PATH) if EICU_PATIENT_PATH else "not found",
                   "column present" if HAVE_EICU_YEAR else "column absent"))

MIMIC4_PATIENTS_PATH, _tried_p = find_first(MIMIC4_PATIENTS_CANDIDATES)
MIMIC4_ICUSTAYS_PATH, _tried_i = find_first(MIMIC4_ICUSTAYS_CANDIDATES)
_cp = {c.lower(): c for c in (header_cols(MIMIC4_PATIENTS_PATH) if MIMIC4_PATIENTS_PATH else [])}
_ci = {c.lower(): c for c in (header_cols(MIMIC4_ICUSTAYS_PATH) if MIMIC4_ICUSTAYS_PATH else [])}
_m4_miss_p = [k for k in ("subject_id", "anchor_year_group") if k not in _cp]
_m4_miss_i = [k for k in ("stay_id", "subject_id") if k not in _ci]
HAVE_MIMIC4_YEAR = bool(MIMIC4_PATIENTS_PATH and MIMIC4_ICUSTAYS_PATH and not _m4_miss_p and not _m4_miss_i)
#   the unit-transfer adjustment works only when patients has anchor_year and icustays has intime.
#   for a copy without anchor_year (anonymized), the periods are split by a patient-level approximation (see read_mimic4_periods and the A1b cell).
M4_HAS_SHIFT = bool(HAVE_MIMIC4_YEAR and "anchor_year" in _cp and "intime" in _ci)
M4_MODE = "unit-transfer adjusted" if M4_HAS_SHIFT else ("patient-level approximation" if HAVE_MIMIC4_YEAR else "absent")
M4_APPROX_WHY = "" if (M4_HAS_SHIFT or not HAVE_MIMIC4_YEAR) else ("anchor_year absent" if "anchor_year" not in _cp else "intime absent")
M4_WHY = []                                  # reason A1b cannot run (human-readable). It runs when empty
if not MIMIC4_PATIENTS_PATH:
    M4_WHY.append("not found: patients")
elif _m4_miss_p:
    M4_WHY.append("column absent: " + ", ".join(_m4_miss_p) + " (patients)")
if not MIMIC4_ICUSTAYS_PATH:
    M4_WHY.append("not found: icustays")
elif _m4_miss_i:
    M4_WHY.append("column absent: " + ", ".join(_m4_miss_i) + " (icustays)")
INPUT_ROWS.append(("MIMIC-IV patients(anchor_year_group)", str(MIMIC4_PATIENTS_PATH) if MIMIC4_PATIENTS_PATH else "not found",
                   ("column present" if not _m4_miss_p else "column absent: " + ", ".join(_m4_miss_p)) if MIMIC4_PATIENTS_PATH else "not found"))
INPUT_ROWS.append(("MIMIC-IV icustays(stay_id · subject_id)", str(MIMIC4_ICUSTAYS_PATH) if MIMIC4_ICUSTAYS_PATH else "not found",
                   ("column present" if not _m4_miss_i else "column absent: " + ", ".join(_m4_miss_i)) if MIMIC4_ICUSTAYS_PATH else "not found"))
INPUT_ROWS.append(("MIMIC-IV period assignment method (A1b)", "unit-transfer adjustment if anchor_year and intime exist, otherwise patient-level approximation", M4_MODE + (" (" + M4_APPROX_WHY + ")" if M4_APPROX_WHY else "")))

try:
    import tabpfn as _tabpfn_mod            # noqa: F401
    HAVE_TABPFN = True
except Exception:  # noqa: BLE001
    HAVE_TABPFN = False
TABPFN_PKG_VERSION = "not installed"
if HAVE_TABPFN:
    try:
        from importlib.metadata import version as _pkg_version
        TABPFN_PKG_VERSION = _pkg_version("tabpfn")
    except Exception:  # noqa: BLE001
        TABPFN_PKG_VERSION = str(getattr(_tabpfn_mod, "__version__", "unknown"))
HAVE_CUDA = False
if HAVE_TABPFN:                              # torch is imported only when TabPFN is present, to look at the GPU (avoids import conflicts of torch in environments without it)
    try:
        import torch as _torch_mod
        HAVE_CUDA = bool(_torch_mod.cuda.is_available())
    except Exception:  # noqa: BLE001
        HAVE_CUDA = False
INPUT_ROWS.append(("tabpfn package", "pip install tabpfn . A9 selects the v2 weights explicitly (create_default_for_version(ModelVersion.V2)). The v2 weights are downloaded on the first run",
                   ("installed " + TABPFN_PKG_VERSION) if HAVE_TABPFN else "absent"))
INPUT_ROWS.append(("GPU(CUDA)", "used by A9 only", "present" if HAVE_CUDA else "absent"))

print("=" * 100)
print("  PRAM profile =", RUN_PROFILE, "(use full when the results go into the paper)" if RUN_PROFILE == "full" else "(for checking only; results are not used in the paper)")
print("=" * 100)
print("| input | path | status |")
print("|---|---|---|")
for _r in INPUT_ROWS:
    print("| %s | %s | %s |" % _r)
if not DATA_OK:
    print("\n  [not a stop] one of the six main inputs is missing. All analysis cells below are skipped. Fix DATA_ROOT and run again ->", DATA_ROOT.resolve())
if not HAVE_HOSPITALID:
    print("\n  The hospital id column was not found. Hospital-level analysis, network and hospital-level temporal analysis are marked not available and skipped. Paths searched:")
    for _t in _tried_e:
        print("    ", _t)
if not HAVE_EICU_YEAR:
    print("\n  The eICU discharge year column (hospitaldischargeyear) was not found. The eICU temporal validation of A1 is skipped as not available.")
if HAVE_MIMIC4_YEAR and not M4_HAS_SHIFT:
    print("\n  The MIMIC-IV period assignment is a patient-level approximation (%s). All stays of the same patient are placed in that patient anchor_year_group period. intime is not used to order patients." % M4_APPROX_WHY)
if not HAVE_MIMIC4_YEAR:
    print("\n  The MIMIC-IV time chain (A1b) cannot run , " + " · ".join(M4_WHY) + ". The MIMIC-IV time chain of A1 is skipped as not available. Paths searched (patients):")
    for _t in _tried_p:
        print("    ", _t)
    print("  paths searched (icustays):")
    for _t in _tried_i:
        print("    ", _t)
print("\n  python %s | numpy %s | pandas %s | sklearn %s" % (platform.python_version(), np.__version__, pd.__version__, sklearn.__version__))
print("  SEED=%d | folds %d | inner folds %d | hospital folds %d | draws %s (tail %s) | bank sizes %s" % (
    SEED, N_FOLDS, N_INNER, N_SITE_SPLITS, N_DRAWS_MAIN, N_DRAWS_TAIL, BANK_SIZES))
print("  output folder %s | zip %s" % (OUT_DIR.resolve(), ZIP_NAME))


In [ ]:
# ================================================================
# PRAM closed testing  new settings of this notebook (existing constants use those of cell 1 as they are)
# ================================================================
CT_ALPHA = CLOSED_TEST_ALPHA                 # significance level: the same single value as the variant (cell 1)
CT_MLE_MAXIT = 25                            # iteration cap of the unpenalized maximum likelihood (R glm.control default 25)
CT_MLE_EPS = 1e-8                            # convergence criterion: relative change in deviance |dev - dev_old| / (|dev| + 0.1) (R glm.control default 1e-8)
CT_RANK_TOL = 1e-10                          # a direction is treated as non-identifiable if the singular value of the design matrix [1, X] is at or below this fraction of the largest singular value
CT_SEP_EPS = 10 * float(np.finfo(float).eps)     # marked complete separation if the fitted probability is within this distance of 0 or 1 (warning criterion of R glm)
CT_FS = [f for f in ("F42", "F33") if f in FS_RUN]
CT_NEW_NAMES = tuple([f"ctpub_{f}" for f in CT_FS] + [f"ctpubinfo_{f}" for f in CT_FS]
                     + [f"ctpubhosp_{f}" for f in CT_FS] + [f"ctpubhospfold_{f}" for f in CT_FS])     # all resume names newly written by this notebook
CT_CSV_PREFIX = "CTPUB_"                     # prefix of new csv files (a csv without this prefix cannot be written)
CT_SUM_NAME = "PRAM-closed-testing-summary.md"
CT_RES_NAME = "PRAM-closed-testing-results.md"
CT_ZIP_NAME = OUT_NAME + "_ctpub.zip"
CT_STEP_LABEL = {0: "no_update", 1: "intercept", 2: "recalibration", 3: "revision"}
print("closed-testing settings: significance level %s (same as the variant) | iteration cap of the unpenalized revision fit %d, convergence %.0e | target feature sets %s" % (CT_ALPHA, CT_MLE_MAXIT, CT_MLE_EPS, CT_FS))
print("  resume names newly written: %s" % (list(CT_NEW_NAMES),))
print("  new zip: %s" % CT_ZIP_NAME)


## Definitions

Preprocessors (`PreprocessorGeneral`, `PreprocessorCatBoost`), data loading (`_load_x`, `_load_y`, `load_mimic4`, `load_external`), cosine retrieval (`retrieval_R1_cosine`), `compute_ece`, `_logit`, the DeLong functions (`_midrank`, `_delong_core_original`), `inner_cv_bank`, `sample_bank`, the anchored ridge (`_anchored_ridge`, `_pick_lam`) and `Resumable`, the store of finished units.

| Definition | Note |
|---|---|
| `load_external`, `load_mimic4` | `load_external` also returns `stay_ids`. `load_mimic4` counts outcomes that are missing and fills them with 0 (a warning is printed) |
| `_midrank` | uses `scipy.stats.rankdata`, which gives the same values as a loop implementation |
| `_delong_core_original` | DeLong core in which `np.arange` is used in place of the within-class midranks. It overestimates the variance of the paired AUROC difference when scores within a class are not sorted (see A6 for its size in the hospital-level data). It is used only by diagnostic cells; every analysis uses the standard fastDeLong `_delong_core` defined with the new functions |
| `inner_cv_bank` | the number of folds defaults to `N_INNER`. Retrieval rule R1 (cosine) does not use mutual information, so no mutual-information computation is made |
| `_anchored_ridge`, `_pick_lam` | the number of folds is `N_INNER` |
| `Resumable` | stores finished units under the resume folder (`out_pram/resume/`) |


In [ ]:
# ================================================================
# PRAM definitions: preprocessors, data loading, retrieval, DeLong, inner cross-validation, anchored ridge, Resumable
# ================================================================
def _log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def stable_seed(*parts):
    s = "|".join(str(p) for p in parts) + f"|SEED={SEED}"
    return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:8], 16) % (2**31 - 1)


def rng_of(*parts):
    return np.random.RandomState(stable_seed(*parts))


def _jsafe(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (Path,)):
        return str(o)
    return str(o)


class Resumable:
    """append per row + completion marker per unit. Even if the kernel dies, units already completed are not recomputed."""

    def __init__(self, name):
        self.name = name
        self.rows_path = RESUME_DIR / f"{name}.jsonl"
        self.units_path = RESUME_DIR / f"{name}.units.jsonl"
        self.rows, self.units = [], set()
        self.n_dropped = 0                                       # counts the incomplete lines that are discarded
        if self.rows_path.exists():
            with open(self.rows_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        self.rows.append(json.loads(line))
                    except json.JSONDecodeError:
                        self.n_dropped += 1                      # discard the incomplete last line at the time of interruption
            if self.n_dropped:
                print(f"  [{name}] incomplete line {self.n_dropped} lines discarded (last line at the time of interruption)")
        if self.units_path.exists():
            with open(self.units_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        self.units.add(line)

    @staticmethod
    def _ukey(unit):
        return "::".join(str(x) for x in unit)

    def unit_done(self, unit):
        return self._ukey(unit) in self.units

    def add_row(self, row):
        with open(self.rows_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row, default=_jsafe) + "\n")
            f.flush()
        self.rows.append(row)

    def mark_unit(self, unit):
        k = self._ukey(unit)
        with open(self.units_path, "a", encoding="utf-8") as f:
            f.write(k + "\n")
            f.flush()
        self.units.add(k)

    def df(self):
        return pd.DataFrame(self.rows) if self.rows else pd.DataFrame()


class PreprocessorGeneral:
    def __init__(self, cat_cols, num_cols, random_state=SEED):
        self.cat_cols, self.num_cols = cat_cols, num_cols
        self.cat_imputer = SimpleImputer(strategy="most_frequent")
        self.ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        self.num_imputer = IterativeImputer(max_iter=10, random_state=random_state, sample_posterior=False)
        self.scaler = StandardScaler()
        self._fitted = False
        self._all_nan_cols = []

    def fit(self, X_df):
        if len(self.cat_cols) > 0:
            Xi = self.cat_imputer.fit_transform(X_df[self.cat_cols].values)
            self.ohe.fit(Xi)
        if len(self.num_cols) > 0:
            X_num = X_df[self.num_cols].values
            self._all_nan_cols = [i for i in range(X_num.shape[1]) if np.all(np.isnan(X_num[:, i]))]
            if self._all_nan_cols:
                X_num = X_num.copy()
                X_num[:, self._all_nan_cols] = 0.0
            Xni = self.num_imputer.fit_transform(X_num)
            self.scaler.fit(Xni)
        self._fitted = True
        return self

    def transform(self, X_df):
        parts = []
        if len(self.cat_cols) > 0:
            parts.append(self.ohe.transform(self.cat_imputer.transform(X_df[self.cat_cols].values)))
        if len(self.num_cols) > 0:
            X_num = X_df[self.num_cols].values.copy()
            if self._all_nan_cols:
                X_num[:, self._all_nan_cols] = 0.0
            parts.append(self.scaler.transform(self.num_imputer.transform(X_num)))
        return np.hstack(parts).astype(np.float32) if parts else np.empty((len(X_df), 0), dtype=np.float32)

    def fit_transform(self, X_df):
        self.fit(X_df)
        return self.transform(X_df)

    def column_names(self):
        """column names after the transform (A10). An auxiliary function; not used in any computation."""
        names = []
        if len(self.cat_cols) > 0:
            names += [str(n) for n in self.ohe.get_feature_names_out(self.cat_cols)]
        names += list(self.num_cols)
        return names


class PreprocessorCatBoost:
    def __init__(self, cat_cols, num_cols, random_state=SEED):
        self.cat_cols, self.num_cols = cat_cols, num_cols
        self.cat_imputer = SimpleImputer(strategy="most_frequent")
        self.num_imputer = IterativeImputer(max_iter=10, random_state=random_state, sample_posterior=False)
        self.cat_feature_indices = list(range(len(cat_cols)))
        self._all_nan_cols = []

    def fit(self, X_df):
        if len(self.cat_cols) > 0:
            self.cat_imputer.fit(X_df[self.cat_cols].values)
        if len(self.num_cols) > 0:
            X_num = X_df[self.num_cols].values
            self._all_nan_cols = [i for i in range(X_num.shape[1]) if np.all(np.isnan(X_num[:, i]))]
            if self._all_nan_cols:
                X_num = X_num.copy()
                X_num[:, self._all_nan_cols] = 0.0
            self.num_imputer.fit(X_num)
        return self

    def transform(self, X_df):
        parts = []
        if len(self.cat_cols) > 0:
            parts.append(self.cat_imputer.transform(X_df[self.cat_cols].values))
        if len(self.num_cols) > 0:
            X_num = X_df[self.num_cols].values.copy()
            if self._all_nan_cols:
                X_num[:, self._all_nan_cols] = 0.0
            parts.append(self.num_imputer.transform(X_num))
        if not parts:
            return pd.DataFrame()
        out = pd.DataFrame(np.hstack(parts), columns=self.cat_cols + self.num_cols)
        for c in self.cat_cols:
            out[c] = out[c].round().astype(int)
        return out

    def fit_transform(self, X_df):
        self.fit(X_df)
        return self.transform(X_df)


# ---- data loading ------------------------------------------
def _detect_cols(X_df):
    cat = [c for c in X_df.columns if X_df[c].dropna().isin([0, 1]).all() and X_df[c].nunique(dropna=True) <= 2]
    num = [c for c in X_df.columns if c not in cat]
    return cat, num


def _load_x(db_name):
    """reads the feature matrix and returns (X without stay_id and fold_id, stay_id array)."""
    if db_name == "mimic4":
        fp = DATA_ROOT / "mimic4" / "X_folded.parquet"
    else:
        fp = DATA_ROOT / db_name / "X_external.parquet"
    X_f = pd.read_parquet(fp)
    stay_ids = X_f["stay_id"].values
    X_df = X_f.drop(columns=[c for c in ["stay_id", "fold_id"] if c in X_f.columns])
    X_df = X_df.drop(columns=[c for c in DROP_COLS if c in X_df.columns])
    return X_df, stay_ids


def _load_y(db_name):
    fp = DATA_ROOT / db_name / f"binary_outcomes_{db_name}.csv"
    return pd.read_csv(fp)


def split_56_14_30(y):
    """70/30 stratified split, then the 70 part is split 80/20 stratified. Train 56, validation 14, test 30."""
    idx_all = np.arange(len(y))
    idx_tr70, idx_te, _, _ = train_test_split(idx_all, y, test_size=0.30, stratify=y, random_state=SEED)
    idx_tr, idx_va, _, _ = train_test_split(idx_tr70, y[idx_tr70], test_size=0.20, stratify=y[idx_tr70], random_state=SEED)
    return idx_tr, idx_va, idx_te


def load_mimic4(outcome="AKI_168h"):
    X_df, stay_ids = _load_x("mimic4")
    y_df = _load_y("mimic4")
    y_col = OUTCOME_MAP[outcome]
    merged = pd.DataFrame({"stay_id": stay_ids}).merge(y_df[["stay_id", y_col]], on="stay_id", how="left")
    n_miss = int(merged[y_col].isna().sum())                     # the number of missing outcomes is counted; they are filled with 0
    if n_miss:
        print(f"  [warning] MIMIC-IV {outcome}: stays without outcome {n_miss} rows were filled with 0 (missing outcomes are filled with 0).")
    y = merged[y_col].fillna(0).values.astype(int)
    cat_cols, num_cols = _detect_cols(X_df)
    idx_tr, idx_va, idx_te = split_56_14_30(y)
    return {"X_df": X_df, "y": y, "stay_ids": stay_ids,
            "idx_train": idx_tr, "idx_val": idx_va, "idx_test": idx_te,
            "cat_cols": cat_cols, "num_cols": num_cols,
            "name": "MIMIC-IV", "outcome": outcome, "n_outcome_missing": n_miss}


def load_external(db_name, outcome):
    X_ref, _ = _load_x("mimic4")
    cat_cols, num_cols = _detect_cols(X_ref)
    X_df, stay_ids = _load_x(db_name)
    y_df = _load_y(db_name)
    y_col = OUTCOME_MAP[outcome]
    X_aligned = pd.DataFrame(index=range(len(stay_ids)))
    for col in cat_cols + num_cols:
        X_aligned[col] = X_df[col].values if col in X_df.columns else np.nan
    merged = X_aligned.copy()
    merged["stay_id"] = stay_ids
    merged = merged.merge(y_df[["stay_id", y_col]], on="stay_id", how="inner")
    y = merged[y_col].values.astype(int)
    stay_out = merged["stay_id"].values                          # stay_id is kept after the merge (for linking to time and hospital)
    X_out = merged.drop(columns=["stay_id", y_col])
    missing_cols = [c for c in cat_cols + num_cols if c not in X_df.columns]
    if missing_cols:
        _log(f"  {db_name}: structurally missing -> {missing_cols}")
    display_name = {"mimic3": "MIMIC-III", "eicu": "eICU"}[db_name]
    return {"X_df": X_out, "y": y, "stay_ids": stay_out, "cat_cols": cat_cols, "num_cols": num_cols,
            "name": display_name, "outcome": outcome}


# ---- retrieval and helper functions ----------------
def retrieval_R1_cosine(c, k=K_NEIGHBORS):
    k_ = min(k, len(c["X_tr_s"]))
    if k_ < 1:
        return np.full(len(c["X_te_s"]), 0.5)
    nn = NearestNeighbors(n_neighbors=k_, metric="cosine", algorithm="brute", n_jobs=-1)
    nn.fit(c["X_tr_s"])
    d, idx = nn.kneighbors(c["X_te_s"])
    w = 1.0 / (d + 1e-6)
    return (w * c["y_tr"][idx]).sum(1) / w.sum(1)


RETRIEVAL_FN = retrieval_R1_cosine


def compute_ece(y, p, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (p >= lo) & (p < hi)
        if m.sum() == 0:
            continue
        ece += m.sum() * abs(y[m].mean() - p[m].mean())
    return ece / len(y)


def _logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


# ---- DeLong -----------------------------------------------
def _midrank(x):
    """Compute midranks for tied values (scipy implementation; gives the same values as a loop implementation)."""
    return scipy_stats.rankdata(np.asarray(x, dtype=float), method="average")


def _delong_core_original(preds_sorted_T, m):
    """The DeLong core that uses np.arange in place of the within-class midranks (the original core). Defect: if the scores within a class
    are not in sorted order, the variance of the paired AUROC difference is overestimated. It is correct only when sorted by score within each class.
    All analyses of this notebook use `_delong_core` below; this function is called only by diagnostic cells that give the original variance for comparison."""
    n = preds_sorted_T.shape[1] - m
    k = preds_sorted_T.shape[0]
    aucs = np.zeros(k)
    v01 = np.zeros((k, m))
    v10 = np.zeros((k, n))
    for i in range(k):
        tx = _midrank(preds_sorted_T[i, :])
        aucs[i] = (np.sum(tx[:m]) - m * (m + 1) / 2.0) / (m * n)
        v10[i] = (tx[m:] - np.arange(1, n + 1)) / m
        v01[i] = 1.0 - (tx[:m] - np.arange(1, m + 1)) / n
    if m > 1:
        S01 = np.cov(v01) if k > 1 else np.array([[np.var(v01[0], ddof=1)]])
    else:
        S01 = np.zeros((k, k))
    if n > 1:
        S10 = np.cov(v10) if k > 1 else np.array([[np.var(v10[0], ddof=1)]])
    else:
        S10 = np.zeros((k, k))
    S = S01 / m + S10 / n
    return aucs, S


def _delong_core(preds_sorted_T, m):
    """Standard fastDeLong (Sun and Xu 2014). Builds the structural components from midranks within positives and negatives (tx, ty) and overall midranks (tz).
    preds_sorted_T is (k, n_total), positives first. Returns (AUROCs, covariance matrix). An internal check compares with the O(mn) definition and with a bootstrap."""
    n = preds_sorted_T.shape[1] - m
    k = preds_sorted_T.shape[0]
    pos, neg = preds_sorted_T[:, :m], preds_sorted_T[:, m:]
    tx = np.array([_midrank(pos[i]) for i in range(k)])
    ty = np.array([_midrank(neg[i]) for i in range(k)])
    tz = np.array([_midrank(preds_sorted_T[i]) for i in range(k)])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    S01 = np.cov(v01) if k > 1 else np.array([[np.var(v01[0], ddof=1)]])
    S10 = np.cov(v10) if k > 1 else np.array([[np.var(v10[0], ddof=1)]])
    return aucs, S01 / m + S10 / n


# ---- mixing-weight selection within a bank and stratified draw ----------------------
def inner_cv_bank(X_bank, y_bank, p_base_bank, r_fn, n_splits=N_INNER, key=""):
    """Select (a) alpha and (b) produce OOF PRAM predictions by cross-validation inside the bank. OOF predictions avoid self-retrieval.
    compute_mi is not called, because R1 cosine does not read mutual information."""
    n = len(y_bank)
    if n < n_splits * 5 or len(np.unique(y_bank)) < 2:
        return {"alpha": 0.0, "oof_pram": p_base_bank.copy(), "ok": False}
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                          random_state=stable_seed("innercv", key))
    scores = {a: [] for a in ALPHA_GRID}
    oof_retr = np.full(n, np.nan)
    for itr, iva in skf.split(np.zeros(n), y_bank):
        mi_i, mp_i = None, None
        pr = r_fn({"X_tr_s": X_bank[itr], "y_tr": y_bank[itr],
                   "X_te_s": X_bank[iva], "y_te": y_bank[iva],
                   "mi": mi_i, "mi_prob": mp_i})
        oof_retr[iva] = pr
        if len(np.unique(y_bank[iva])) > 1:
            for a in ALPHA_GRID:
                scores[a].append(roc_auc_score(y_bank[iva],
                                               (1 - a) * p_base_bank[iva] + a * pr))
    best_a = max(ALPHA_GRID, key=lambda a: np.nanmean(scores[a]) if scores[a] else -1)
    oof_retr = np.where(np.isnan(oof_retr), p_base_bank, oof_retr)
    return {"alpha": best_a, "oof_pram": (1 - best_a) * p_base_bank + best_a * oof_retr,
            "ok": True}


def sample_bank(pool_y, size, key):
    """stratified draw. Uses round()."""
    r = rng_of("bank", key)
    if size == "full":
        return np.arange(len(pool_y))
    prev = pool_y.mean()
    n_pos = max(1, int(round(size * prev)))
    n_neg = max(1, size - n_pos)
    pi, ni = np.where(pool_y == 1)[0], np.where(pool_y == 0)[0]
    sp = r.choice(pi, min(n_pos, len(pi)), replace=len(pi) < n_pos)
    sn = r.choice(ni, min(n_neg, len(ni)), replace=len(ni) < n_neg)
    return np.sort(np.concatenate([sp, sn]))


# ---- anchored ridge -------------------------------------------
def _anchored_ridge(X, y, w_src, b_src, lam, anchor_intercept=True, maxiter=500):
    """minimizes mean log loss + lam*||w - w_src||^2. Returns (w, b, ||u||, converged)"""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, d = X.shape
    off = X @ w_src + b_src

    def fun(t):
        u = t[:d]
        db = 0.0 if anchor_intercept else t[d]
        z = off + X @ u + db
        loss = float(np.mean(np.logaddexp(0.0, z) - y * z)) + lam * float(u @ u)
        g = (1.0 / (1.0 + np.exp(-z)) - y) / n
        gu = X.T @ g + 2.0 * lam * u
        return (loss, gu) if anchor_intercept else (loss, np.concatenate([gu, [g.sum()]]))

    t0 = np.zeros(d if anchor_intercept else d + 1)
    r = minimize(fun, t0, jac=True, method="L-BFGS-B",
                 options={"maxiter": maxiter, "ftol": 1e-12, "gtol": 1e-9})
    u = r.x[:d]
    db = 0.0 if anchor_intercept else float(r.x[d])
    return w_src + u, b_src + db, float(np.sqrt(u @ u)), bool(r.success)


def _pick_lam(Xb, yb, w_src, b_src, anchor_intercept, key, n_splits=N_INNER):
    """lam is chosen within the bank only."""
    if len(np.unique(yb)) < 2 or len(yb) < n_splits * 5:
        return float(LAM_GRID[-1])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                          random_state=stable_seed("a53", key))
    scores = {float(l): [] for l in LAM_GRID}
    for itr, iva in skf.split(np.zeros(len(yb)), yb):
        if len(np.unique(yb[iva])) < 2:
            continue
        for l in LAM_GRID:
            w, b, _, _ = _anchored_ridge(Xb[itr], yb[itr], w_src, b_src,
                                         float(l), anchor_intercept, maxiter=200)
            scores[float(l)].append(roc_auc_score(yb[iva], Xb[iva] @ w + b))
    best = max(scores, key=lambda l: np.nanmean(scores[l]) if scores[l] else -1)
    return float(best)


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))


## New functions

Every analysis of this notebook calls one engine (`bank_predictions`) that returns the test predictions of the strategies for one bank and one set of test patients, so the main analysis, temporal validation, hospital level and tail risk share the same code.

| Definition | What |
|---|---|
| `full_metrics` | AUROC, AUPRC, Brier score, ECE, calibration slope and intercept, and net benefit over the full 1 to 50% threshold grid. The slope is an unpenalized Newton fit |
| `update_hierarchy` | update steps 0 to 3 and closed testing, on the frozen logistic model only: intercept only (1, maximum likelihood), intercept and slope (2, unpenalized), model revision (3). The test is a likelihood-ratio test of the unpenalized revision; when step 3 is chosen, the prediction is the **anchored ridge** with the penalty chosen within the bank. The order of the closed-testing steps follows Vergouwe et al. (2017) |
| `bank_predictions` | test predictions per strategy group for one bank and one set of test patients |
| `run_tail_unit` | tail risk: one (bank, test fold) pair; 5 folds x 20 draws per fold |
| `run_temporal` | temporal validation: the test period is split into 2 folds that alternate as test; the past bank (P) and the same-period bank (S) are compared at the same size |
| `hospital_oof`, `hospital_unit` | one hospital, 3 folds |
| `network_unit` | an unlabeled hospital receives the mean probability of its partner hospitals |
| `meta_reml_hk` | REML tau squared, Hartung-Knapp interval (untruncated and truncated forms), prediction interval, inverse-variance I squared, Q and degrees of freedom |
| `delong_delta_var`, `_delong_core` | paired AUROC difference and its variance (standard fastDeLong, Sun and Xu 2014). Checked against the O(mn) definition and a bootstrap |


In [ ]:
# ================================================================
# PRAM Cell 3. Newly written definitions (reporting, metrics, update hierarchy)
# ================================================================
try:
    from IPython.display import display, Markdown
    from IPython import get_ipython as _get_ipy
    _IN_IPY = _get_ipy() is not None
except Exception:  # noqa: BLE001
    _IN_IPY = False

REPORT_LINES = []                       # every table shown on screen is also accumulated in the result md


def emit(text=""):
    REPORT_LINES.append(str(text))
    if _IN_IPY:
        display(Markdown(str(text)))
    else:
        print(text)


def md_table(df, digits=4, max_rows=80):
    if df is None or len(df) == 0:
        return "(empty)"
    d = df.head(max_rows)
    cols = [str(c) for c in d.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in d.itertuples(index=False):
        cells = []
        for v in r:
            if isinstance(v, (float, np.floating)):
                cells.append("" if not np.isfinite(v) else f"{v:.{digits}f}")
            else:
                cells.append(str(v))
        lines.append("| " + " | ".join(cells) + " |")
    if len(df) > max_rows:
        lines.append(f"... {len(df) - max_rows} rows omitted (full table in csv)")
    return "\n".join(lines)


def emit_table(title, df, digits=4, max_rows=80):
    emit("\n**" + title + "**\n")
    emit(md_table(df, digits=digits, max_rows=max_rows))


def save_csv(df, name):
    path = CSV_DIR / (name + ".csv")
    df.to_csv(path, index=False, encoding="utf-8-sig", float_format="%.6g")
    emit(f"- saved `{path.as_posix()}` ({len(df)}rows)")
    return path


# ---- metrics ------------------------------------------------------------------
NB_COLS = [f"nb@{float(t)}" for t in THRESH_GRID]


def _newton_2(y, x, iters=60):
    """unpenalized maximum likelihood of p = sigmoid(a + b x). Starts at (0, 1). Returns (nan, nan) on failure."""
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    if (not np.isfinite(x).all()) or x.std() < 1e-9 or len(np.unique(y)) < 2:
        return np.nan, np.nan

    def nll(a_, b_):
        z = a_ + b_ * x
        return float(np.sum(np.logaddexp(0.0, z) - y * z))

    a, b = 0.0, 1.0
    cur = nll(a, b)
    for _ in range(iters):
        pr = _sigmoid(a + b * x)
        w = pr * (1 - pr) + 1e-12
        g = np.array([np.sum(y - pr), np.sum((y - pr) * x)])
        H = np.array([[w.sum(), (w * x).sum()], [(w * x).sum(), (w * x * x).sum()]]) + 1e-10 * np.eye(2)
        step = np.linalg.solve(H, g)
        t = 1.0
        while t > 1e-4 and nll(a + t * step[0], b + t * step[1]) > cur + 1e-12:
            t *= 0.5
        a, b = a + t * step[0], b + t * step[1]
        new = nll(a, b)
        done = abs(cur - new) < 1e-10 or max(abs(t * step[0]), abs(t * step[1])) < 1e-9
        cur = new
        if done:
            break
    return float(a), float(b)


def calib_slope_intercept(y, p):
    """(slope, intercept). Unpenalized calibration line on the logit."""
    a, b = _newton_2(y, _logit(np.asarray(p, dtype=float)))
    return b, a


def nb_curve(y, p, grid=THRESH_GRID):
    """net benefit over the whole threshold grid. One sort counts all thresholds."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    n = len(y)
    order = np.argsort(-p, kind="stable")
    ps, ys = p[order], y[order]
    ctp = np.cumsum(ys)
    cfp = np.cumsum(1 - ys)
    k = np.searchsorted(-ps, -np.asarray(grid, dtype=float), side="right")      # number of patients with p >= t
    tp = np.where(k > 0, ctp[np.maximum(k - 1, 0)], 0)
    fp = np.where(k > 0, cfp[np.maximum(k - 1, 0)], 0)
    g = np.asarray(grid, dtype=float)
    return tp / n - fp / n * (g / (1 - g))


def full_metrics(y, p, orig_slope=False):
    """if orig_slope=True, adds the slope of a scikit-learn C=1e6 logistic fit as slope_orig. For reproduction checks only."""
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    out = {}
    try:
        out["auroc"] = float(roc_auc_score(y, p))
        out["auprc"] = float(average_precision_score(y, p))
        out["brier"] = float(brier_score_loss(y, p))
    except ValueError:
        out["auroc"] = out["auprc"] = out["brier"] = np.nan
    out["ece"] = float(compute_ece(y, p))
    sl, ic = calib_slope_intercept(y, p)
    out["slope"], out["intercept"] = sl, ic
    if orig_slope:
        lp = _logit(p)
        out["slope_orig"] = float(LogisticRegression(max_iter=1000, C=1e6).fit(lp.reshape(-1, 1), y).coef_[0][0])             if (np.isfinite(lp).all() and lp.std() > 1e-9 and len(np.unique(y)) > 1) else np.nan
    out["mean_pred"] = float(p.mean())
    out["obs_rate"] = float(y.mean())
    for c, v in zip(NB_COLS, nb_curve(y, p)):
        out[c] = float(v)
    return out


def delong_delta_var(y, pa, pb, original=False):
    """paired DeLong. (AUROC_a, AUROC_b, difference, variance of the difference). nan if one class is empty. If original=True, the original implementation (for diagnosis)."""
    y = np.asarray(y).astype(int)
    if len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan, np.nan
    order = np.argsort(y)[::-1]
    m = int(y.sum())
    if m < 2 or len(y) - m < 2:
        return np.nan, np.nan, np.nan, np.nan
    preds = np.vstack([np.asarray(pa, dtype=float)[order], np.asarray(pb, dtype=float)[order]])
    aucs, S = (_delong_core_original if original else _delong_core)(preds, m)
    var = float(S[0, 0] + S[1, 1] - 2 * S[0, 1])
    return float(aucs[0]), float(aucs[1]), float(aucs[0] - aucs[1]), max(var, 0.0)


def cp_interval(k, n, level=CP_LEVEL):
    """Clopper-Pearson exact interval."""
    if n <= 0:
        return np.nan, np.nan
    a = 1 - level
    lo = 0.0 if k == 0 else scipy_stats.beta.ppf(a / 2, k, n - k + 1)
    hi = 1.0 if k == n else scipy_stats.beta.ppf(1 - a / 2, k + 1, n - k)
    return float(lo), float(hi)


def area_under_curve(grid, vals, lo, hi):
    """trapezoid area within [lo, hi]."""
    g = np.asarray(grid, dtype=float)
    v = np.asarray(vals, dtype=float)
    m = (g >= lo - 1e-9) & (g <= hi + 1e-9)
    if m.sum() < 2:
        return np.nan
    trap = getattr(np, "trapezoid", None) or np.trapz
    return float(trap(v[m], g[m]))


# ---- update hierarchy and closed testing (A4) ---------------------------------------------
def _nll_sum(y, z):
    return float(np.sum(np.logaddexp(0.0, z) - y * z))


def fit_intercept_only(y, off):
    """re-estimates the intercept only (recalibration-in-the-large). The slope is fixed at 1."""
    a = 0.0
    for _ in range(60):
        pr = _sigmoid(off + a)
        step = float(np.sum(y - pr) / (np.sum(pr * (1 - pr)) + 1e-12))
        step = max(-5.0, min(5.0, step))
        a += step
        if abs(step) < 1e-10:
            break
    return a


def fit_revision(X, y, off, ridge=REVISION_RIDGE):
    """model revision: the frozen linear predictor is an offset; the intercept and all coefficients are re-estimated. The tiny regularization is for numerical stability."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, d = X.shape

    def fun(t):
        a, u = t[0], t[1:]
        z = off + a + X @ u
        loss = float(np.mean(np.logaddexp(0.0, z) - y * z)) + ridge * float(u @ u)
        g = (_sigmoid(z) - y) / n
        return loss, np.concatenate([[g.sum()], X.T @ g + 2.0 * ridge * u])

    r = minimize(fun, np.zeros(d + 1), jac=True, method="L-BFGS-B",
                 options={"maxiter": 300, "ftol": 1e-12, "gtol": 1e-8})
    return float(r.x[0]), r.x[1:].copy(), bool(r.success)


def update_hierarchy(Xb, yb, offb, Xt, offt, p_frozen_t, p_rev_alt=None):
    """fits update steps 0 to 3 in the bank and chooses one step by closed testing.
    0 no update | 1 intercept only | 2 intercept + slope | 3 model revision. Closed testing (likelihood ratio, significance level CLOSED_TEST_ALPHA):
    (1) if revision versus no update (d+1 degrees of freedom) is not significant, 0. (2) if significant and revision versus step 2 (d-1 degrees of freedom) is significant, 3.
    (3) otherwise, if step 2 versus step 1 (1 degree of freedom) is significant, 2, otherwise 1.
    if p_rev_alt is given, its prediction (anchored ridge with the penalty chosen within the bank) is used when step 3 is chosen. The test always uses the likelihood of the unpenalized revision."""
    res = {"upd1": p_frozen_t, "upd2": p_frozen_t, "upd3": p_frozen_t, "ct": p_frozen_t,
           "step": 0, "p_all": np.nan, "p_rev_vs_recal": np.nan, "p_slope": np.nan,
           "slope2": np.nan, "ok": False}
    yb = np.asarray(yb).astype(float)
    n1 = int(yb.sum())
    n0 = int(len(yb) - n1)
    if min(n1, n0) < N_EVENT_MIN or not np.isfinite(offb).all() or not np.isfinite(offt).all():
        return res
    d = Xb.shape[1]
    ll0 = -_nll_sum(yb, offb)
    a1 = fit_intercept_only(yb, offb)
    ll1 = -_nll_sum(yb, offb + a1)
    a2, b2 = _newton_2(yb, offb)
    if not np.isfinite(a2):
        return res
    ll2 = -_nll_sum(yb, a2 + b2 * offb)
    a3, u3, _ok3 = fit_revision(Xb, yb, offb)
    ll3 = -_nll_sum(yb, offb + a3 + Xb @ u3)
    res["upd1"] = _sigmoid(offt + a1)
    res["upd2"] = _sigmoid(a2 + b2 * offt)
    res["upd3"] = _sigmoid(offt + a3 + Xt @ u3)
    res["slope2"] = float(b2)
    lr_all = max(0.0, 2 * (ll3 - ll0))
    lr_rev = max(0.0, 2 * (ll3 - ll2))
    lr_slope = max(0.0, 2 * (ll2 - ll1))
    res["p_all"] = float(scipy_stats.chi2.sf(lr_all, d + 1))
    res["p_rev_vs_recal"] = float(scipy_stats.chi2.sf(lr_rev, max(d - 1, 1)))
    res["p_slope"] = float(scipy_stats.chi2.sf(lr_slope, 1))
    if res["p_all"] >= CLOSED_TEST_ALPHA:
        step = 0
    elif res["p_rev_vs_recal"] < CLOSED_TEST_ALPHA:
        step = 3
    elif res["p_slope"] < CLOSED_TEST_ALPHA:
        step = 2
    else:
        step = 1
    res["step"] = step
    res["ct"] = [p_frozen_t, res["upd1"], res["upd2"], res["upd3"] if p_rev_alt is None else p_rev_alt][step]
    res["ok"] = True
    return res


# ---- auxiliary fits -----------------------------------------------------------
def platt_sklearn(p_fit, y_fit, p_apply):
    """Platt scaling: logistic regression with default regularization on the logit."""
    lr_p = LogisticRegression(max_iter=1000, random_state=SEED)
    lr_p.fit(_logit(p_fit).reshape(-1, 1), y_fit)
    return lr_p.predict_proba(_logit(p_apply).reshape(-1, 1))[:, 1]


def _logloss(y, p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def inner_oof_candidates(Xb, yb, p1b, p5b, icv1, icv5, key, cand):
    """A11: builds the within-bank predictions of each candidate strategy by inner cross-validation. Only strategies that a hospital can actually choose are candidates."""
    out = {}
    if "b1_base" in cand:
        out["b1_base"] = p1b
    if "b5_base" in cand:
        out["b5_base"] = p5b
    if "pram_b1" in cand:
        out["pram_b1"] = icv1["oof_pram"]
    if "pram_b5" in cand:
        out["pram_b5"] = icv5["oof_pram"]
    need_fit = [c for c in ("platt_b1", "recal_iso", "local_lr") if c in cand]
    if need_fit:
        oofs = {c: p1b.copy() for c in need_fit}
        skf = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=stable_seed("selcv", key))
        for itr, iva in skf.split(np.zeros(len(yb)), yb):
            if len(np.unique(yb[itr])) < 2:
                continue
            if "platt_b1" in oofs:
                oofs["platt_b1"][iva] = platt_sklearn(p1b[itr], yb[itr], p1b[iva])
            if "recal_iso" in oofs:
                iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip").fit(p1b[itr], yb[itr])
                oofs["recal_iso"][iva] = iso.predict(p1b[iva])
            if "local_lr" in oofs:
                lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xb[itr], yb[itr])
                oofs["local_lr"][iva] = lr.predict_proba(Xb[iva])[:, 1]
        out.update(oofs)
    return out


# ---- strategy engine -----------------------------------------------------------------
def bank_predictions(C, bank, test, key, groups, cb_iter=None):
    """test predictions of the strategies fitted on one bank (row indices). Same rule as run_matrix_unit.
    C is the target cohort context (build_ctx). groups: frozen pram recal local localcb anch hier combo sel."""
    cb_iter = LOCAL_CB_ITER if cb_iter is None else cb_iter
    Xg, y, pb1, pb5 = C["Xg"], C["y"], C["pb1"], C["pb5"]
    Xb, yb, p1b, p5b = Xg[bank], y[bank], pb1[bank], pb5[bank]
    Xt, yt, p1t, p5t = Xg[test], y[test], pb1[test], pb5[test]
    P = {"b1_base": p1t, "b5_base": p5t}
    info = {}
    two = len(np.unique(yb)) > 1
    icv1 = icv5 = p_retr = None

    if "pram" in groups or "combo" in groups or "sel" in groups:
        p_retr = RETRIEVAL_FN({"X_tr_s": Xb, "y_tr": yb, "X_te_s": Xt, "y_te": yt, "mi": None, "mi_prob": None})
        icv1 = inner_cv_bank(Xb, yb, p1b, RETRIEVAL_FN, key="b1|" + key)
        icv5 = inner_cv_bank(Xb, yb, p5b, RETRIEVAL_FN, key="b5|" + key)
        P["pram_b1"] = (1 - icv1["alpha"]) * p1t + icv1["alpha"] * p_retr
        P["pram_b5"] = (1 - icv5["alpha"]) * p5t + icv5["alpha"] * p_retr
        info["alpha_b1"], info["alpha_b5"] = icv1["alpha"], icv5["alpha"]

    if "recal" in groups:
        if two:
            iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip").fit(p1b, yb)
            P["recal_iso"] = iso.predict(p1t)
            P["platt_b1"] = platt_sklearn(p1b, yb, p1t)
            P["platt_b5"] = platt_sklearn(p5b, yb, p5t)          # CatBoost gets Platt only (the update hierarchy is defined on the logistic model)
        else:
            P["recal_iso"], P["platt_b1"], P["platt_b5"] = p1t, p1t, p5t

    if "local" in groups:
        if two:
            lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xb, yb)
            P["local_lr"] = lr.predict_proba(Xt)[:, 1]
            pl = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
            pl.fit(np.vstack([C["Xs_src"], Xb]), np.concatenate([C["ys_src"], yb]))
            P["pooled_lr"] = pl.predict_proba(Xt)[:, 1]
        else:
            P["local_lr"], P["pooled_lr"] = p1t, p1t

    if "localcb" in groups:
        if two:
            cb = CatBoostClassifier(iterations=cb_iter, depth=6, learning_rate=0.05, verbose=0, random_seed=SEED)
            cb.fit(C["Xc"].iloc[bank], yb, cat_features=C["cb_cat_idx"])
            P["local_cb"] = cb.predict_proba(C["Xc"].iloc[test])[:, 1]
        else:
            P["local_cb"] = p1t

    if "anch" in groups and two and (ANCHORED_MAX_BANK is None or len(bank) <= ANCHORED_MAX_BANK):
        lam = _pick_lam(Xb, yb, C["w_src"], C["b_src"], True, "anch|" + key)
        w_a, b_a, _, _ = _anchored_ridge(Xb, yb, C["w_src"], C["b_src"], lam, True)
        P["anchored_ridge"] = _sigmoid(Xt @ w_a + b_a)
        info["anch_lam"] = lam

    if "hier" in groups:                       # the update hierarchy only on the frozen logistic model (for CatBoost see platt_b5 above)
        u = update_hierarchy(Xb, yb, _logit(p1b), Xt, _logit(p1t), p1t, p_rev_alt=P.get("anchored_ridge"))
        P["upd1_b1"], P["upd2_b1"], P["upd3_b1"], P["ct_b1"] = u["upd1"], u["upd2"], u["upd3"], u["ct"]
        info["ct_step_b1"] = u["step"]
        info["ct_p_all_b1"] = u["p_all"]
        info["ct_p_rev_vs_recal_b1"] = u["p_rev_vs_recal"]
        info["slope2_b1"] = u["slope2"]

    if "combo" in groups and two:
        # retrieval -> Platt: Platt is fitted on the inner cross-validation OOF mixture predictions in the bank and applied to the test mixture prediction 
        P["pram_platt_b1"] = platt_sklearn(icv1["oof_pram"], yb, P["pram_b1"])
        P["pram_platt_b5"] = platt_sklearn(icv5["oof_pram"], yb, P["pram_b5"])
    elif "combo" in groups:
        P["pram_platt_b1"], P["pram_platt_b5"] = p1t, p5t

    if "sel" in groups:
        cand = [c for c in SEL_CANDIDATES if c in P]
        oof_c = inner_oof_candidates(Xb, yb, p1b, p5b, icv1, icv5, key, cand)
        cand = [c for c in cand if c in oof_c]
        t_site = T_SITE[C["outcome"]]
        ll = {c: _logloss(yb, oof_c[c]) for c in cand}
        nbv = {c: float(nb_curve(yb, oof_c[c], np.array([t_site]))[0]) for c in cand}
        pick_ll = min(cand, key=lambda c: ll[c])
        pick_nb = max(cand, key=lambda c: nbv[c])
        P["sel_logloss"], P["sel_nb"] = P[pick_ll], P[pick_nb]
        info["sel_logloss"], info["sel_nb"] = pick_ll, pick_nb
    return P, info


def run_matrix_unit(C, fs, outcome, target, bank_size, n_draws, groups, key_tag="m23", keep=None, delong_pairs=None, orig_slope=False):
    """one unit (feature set, outcome, target, bank size). For each draw (rep), the 5 folds are pooled and the metrics are computed once per bank result (pooled).
    the draw keys use key_tag m23, so the same patients are drawn in every run.
    if keep is given, only those strategies get metrics. If delong_pairs={strategy: reference}, the paired DeLong difference and its variance for that strategy are attached to the row."""
    y = C["y"]
    n = len(y)
    folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(n), y))
    n_rep = 1 if bank_size == "full" else n_draws
    rows, info_rows = [], []
    for rep in range(n_rep):
        oof = {}
        for fi, (pool_idx, test_idx) in enumerate(folds):
            key = f"{outcome}|{target}|{bank_size}|{fi}|{rep}"
            if bank_size == "full":
                sel = np.arange(len(pool_idx))
            else:
                sel = sample_bank(y[pool_idx], bank_size, key_tag + "|" + key)
            bank = pool_idx[sel]
            P, info = bank_predictions(C, bank, test_idx, key, set(groups))
            for m, v in P.items():
                if m not in oof:
                    oof[m] = np.full(n, np.nan, dtype=np.float32)
                oof[m][test_idx] = v
            info_rows.append({"fs": fs, "outcome": outcome, "target": target, "bank": str(bank_size),
                              "rep": rep, "fold": fi, "n_bank": int(len(bank)),
                              "events_bank": int(y[bank].sum()), **info})
        for m, pm in oof.items():
            if np.isnan(pm).any() or (keep is not None and m not in keep):
                continue
            row = {"fs": fs, "outcome": outcome, "target": target, "bank": str(bank_size),
                   "rep": rep, "method": m, **full_metrics(y, pm.astype(float), orig_slope=orig_slope)}
            if delong_pairs and m in delong_pairs and delong_pairs[m] in oof:
                _a, _b, dd, vv = delong_delta_var(y, pm.astype(float), oof[delong_pairs[m]].astype(float))
                row["d_auroc_base"], row["var_auroc_base"] = dd, vv
            rows.append(row)
    return rows, info_rows


# ---- development model and target context -------------------------------------------------------
def build_dev(X_train_df, y_train, cols, cat_ref, num_ref, tag):
    """fits the two frozen models (logistic and CatBoost) and the preprocessor on the development training data only."""
    cc = [c for c in cat_ref if c in cols]
    nc = [c for c in num_ref if c in cols]
    X_train_df = X_train_df.reset_index(drop=True)
    y_train = np.asarray(y_train).astype(int)
    pg = PreprocessorGeneral(cc, nc)
    Xtr_g = pg.fit_transform(X_train_df)
    pc = PreprocessorCatBoost(cc, nc)
    Xtr_c = pc.fit_transform(X_train_df)
    m1 = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xtr_g, y_train)
    m5 = CatBoostClassifier(iterations=DEV_CB_ITER, depth=6, learning_rate=0.05, verbose=0, random_seed=SEED)
    m5.fit(Xtr_c, y_train, cat_features=pc.cat_feature_indices)
    return {"pg": pg, "pc": pc, "m1": m1, "m5": m5, "Xs_src": Xtr_g, "ys_src": y_train,
            "cb_cat_idx": pc.cat_feature_indices, "cols": cc + nc, "tag": tag,
            "w_src": np.asarray(m1.coef_).ravel().astype(float), "b_src": float(np.asarray(m1.intercept_).ravel()[0])}


def make_ctx(dev, X_df, y, stay_ids, outcome):
    """moves the target cohort through the development preprocessor and attaches the frozen model outputs."""
    X_df = X_df.reset_index(drop=True)
    Xg = dev["pg"].transform(X_df)
    Xc = dev["pc"].transform(X_df)
    return {"Xg": Xg, "Xc": Xc, "y": np.asarray(y).astype(int),
            "pb1": dev["m1"].predict_proba(Xg)[:, 1], "pb5": dev["m5"].predict_proba(Xc)[:, 1],
            "stay_ids": np.asarray(stay_ids), "outcome": outcome,
            "Xs_src": dev["Xs_src"], "ys_src": dev["ys_src"], "cb_cat_idx": dev["cb_cat_idx"],
            "w_src": dev["w_src"], "b_src": dev["b_src"], "d": int(Xg.shape[1])}


# ---- hospital id and discharge year ---------------------------------------------------------
def load_eicu_patient_table():
    """reads only the stay, hospital and discharge year columns of the patient table (those that exist)."""
    cols = [c for c in (EICU_COL_STAY, EICU_COL_HOSP, EICU_COL_YEAR) if c]
    df = pd.read_csv(EICU_PATIENT_PATH, usecols=cols)
    ren = {EICU_COL_STAY: "stay_id"}
    if EICU_COL_HOSP:
        ren[EICU_COL_HOSP] = "hospitalid"
    if EICU_COL_YEAR:
        ren[EICU_COL_YEAR] = "year"
    df = df.rename(columns=ren)
    df["stay_id"] = pd.to_numeric(df["stay_id"], errors="coerce")
    df = df.dropna(subset=["stay_id"]).drop_duplicates("stay_id")
    df["stay_id"] = df["stay_id"].astype("int64")
    return df.set_index("stay_id")


def map_to_stays(table, col, stay_ids):
    s = pd.to_numeric(pd.Series(np.asarray(stay_ids)), errors="coerce")
    return s.map(table[col]).to_numpy(dtype=float)


def eligible_sites(y, hid):
    """eligible hospitals: at least MIN_SITE_N stays, at least MIN_SITE_EVENTS events and non-events each. Largest hospitals first."""
    keep = []
    for h in pd.unique(hid[pd.notna(hid)]):
        m = (hid == h)
        if m.sum() >= MIN_SITE_N and y[m].sum() >= MIN_SITE_EVENTS and (m.sum() - y[m].sum()) >= MIN_SITE_EVENTS:
            keep.append(int(h))
    keep.sort(key=lambda h: -int((hid == h).sum()))
    return keep


# ---- hospital level ----------------------------
def hospital_oof(C, outcome, h, hid, groups):
    """one set of 3-fold test predictions (OOF) of one hospital and the per-fold event counts. (None, None, None) if not eligible."""
    y = C["y"]
    idx = np.where(hid == h)[0]
    ys = y[idx]
    n_sp = min(N_SITE_SPLITS, int(ys.sum()), int(len(ys) - ys.sum()))
    if n_sp < 2:
        return None, None, None
    skf = StratifiedKFold(n_splits=n_sp, shuffle=True, random_state=stable_seed("a29", outcome, int(h)))
    oof, fold_rows = {}, []
    for fi, (bank_i, test_i) in enumerate(skf.split(np.zeros(len(ys)), ys)):
        bank, test = idx[bank_i], idx[test_i]
        yb, yte = y[bank], y[test]
        if len(np.unique(yb)) < 2 or len(np.unique(yte)) < 2:
            return None, None, None
        P, info = bank_predictions(C, bank, test, f"a29|{outcome}|{h}|{fi}", set(groups))
        for m, v in P.items():
            if m not in oof:
                oof[m] = np.full(len(ys), np.nan)
            oof[m][test_i] = v
        fold_rows.append({"outcome": outcome, "hospitalid": int(h), "fold": fi,
                          "n_test": int(len(test)), "events_test": int(yte.sum()),
                          "n_bank": int(len(bank)), "events_bank": int(yb.sum()), **info})
    return ys, oof, fold_rows


def hospital_unit(C, outcome, h, hid, groups):
    ys, oof, fold_rows = hospital_oof(C, outcome, h, hid, groups)
    if ys is None:
        return None, None
    rows = []
    base1, base5 = oof["b1_base"], oof["b5_base"]
    for m, v in oof.items():
        if np.isnan(v).any():
            continue
        _a, _b, d1, v1 = delong_delta_var(ys, v, base1)
        _a, _b, d5, v5 = delong_delta_var(ys, v, base5)
        rows.append({"outcome": outcome, "hospitalid": int(h), "method": m, "n_site": int(len(ys)),
                     "n_events": int(ys.sum()), "prev": float(ys.mean()), **full_metrics(ys, v),
                     "d_auroc_b1": d1, "var_b1": v1, "d_auroc_b5": d5, "var_b5": v5})
    return rows, fold_rows


# ---- network of unlabeled hospitals ------
def _topk_cos(Xb, yb, Xq, k):
    kk = int(min(k, len(Xb)))
    nn = NearestNeighbors(n_neighbors=kk, metric="cosine").fit(Xb)
    d, i = nn.kneighbors(Xq)
    return d, yb[i]


def _site_reply(d, yk):
    """the one probability a hospital returns to the network."""
    w = 1.0 / (d + 1e-6)
    return (w * yk).sum(axis=1) / np.maximum(w.sum(axis=1), 1e-12)


def _resample_prev(gi, y, r, seed):
    """scales the prevalence of the partner bank by a factor r. When raising, all events + some non-events; when lowering, all non-events + some events."""
    rs = np.random.RandomState(seed)
    yg = y[gi]
    pos, neg = gi[yg == 1], gi[yg == 0]
    prev = len(pos) / len(gi)
    q = min(max(prev * r, 0.005), 0.95)
    if q > prev:
        n_neg = max(1, min(int(round(len(pos) * (1 - q) / q)), len(neg)))
        return np.sort(np.concatenate([pos, rs.choice(neg, n_neg, replace=False)]))
    n_pos = max(1, min(int(round(len(neg) * q / (1 - q))), len(pos)))
    return np.sort(np.concatenate([neg, rs.choice(pos, n_pos, replace=False)]))


def network_unit(C, outcome, h, hid, sites, factors, alphas):
    """mixes the mean probability of the partners (up to 20 hospitals) into the 30% test portion of hospital h.
    factors are the partner bank prevalence multipliers. With 1.0 the whole partner cohort is used."""
    Xg, y, pb1 = C["Xg"], C["y"], C["pb1"]
    idx = np.where(hid == h)[0]
    rng = rng_of("a32", outcome, int(h))
    perm = rng.permutation(len(idx))
    n_te = max(60, int(TEST_FRAC_NET * len(idx)))
    te = idx[perm[:n_te]]
    yte = y[te]
    if len(np.unique(yte)) < 2 or len(idx) - n_te < 60:
        return None
    Xte, p1te = Xg[te], pb1[te]
    partners = [g for g in sites if g != h]
    rng.shuffle(partners)
    partners = partners[:MAX_PARTNERS]
    if len(partners) < 4:
        return None
    own_prev = float(y[idx].mean())
    base = full_metrics(yte, p1te)
    nb_c = f"nb@{float(T_SITE[outcome])}"          # column name of the net benefit at the outcome-specific representative threshold (midpoint of the outcome-specific range) (same as nbcol)
    rows = []
    for r in factors:
        P_list, prevs = [], []
        for g in partners:
            gi = np.where(hid == g)[0]
            if r != 1.0:
                gi = _resample_prev(gi, y, r, stable_seed("a7", outcome, int(h), int(g), r))
            d, yk = _topk_cos(Xg[gi], y[gi].astype(float), Xte, K_NEIGHBORS)
            P_list.append(_site_reply(d, yk))
            prevs.append(float(y[gi].mean()))
        p_net = np.column_stack(P_list).mean(axis=1)
        for a in alphas:
            pm = (1 - a) * p1te + a * p_net
            m = full_metrics(yte, pm)
            rows.append({"outcome": outcome, "hospitalid": int(h), "factor": float(r), "alpha": float(a),
                         "n_test": int(len(te)), "n_partners": int(len(partners)),
                         "prev_own": own_prev, "prev_partner": float(np.mean(prevs)),
                         "prev_diff": float(np.mean(prevs)) - own_prev,
                         "d_auroc": m["auroc"] - base["auroc"], "base_auroc": base["auroc"],
                         "base_slope": base["slope"], "base_intercept": base["intercept"],
                         "base_ece": base["ece"], "d_ece": m["ece"] - base["ece"],
                         "rep_threshold": float(T_SITE[outcome]), "base_nb_rep": base[nb_c], "nb_rep": m[nb_c], "d_nb_rep": m[nb_c] - base[nb_c], **m})
    return rows


# ---- meta-analysis (A6) ------------------------------------------------------------
def meta_reml_hk(d, v, exclude_zero_var=True):
    """random-effects meta-analysis of per-hospital estimates d and variances v. REML tau squared, Hartung-Knapp interval (original and truncated forms: q lower-bounded at 1),
    95% prediction interval (t distribution, k-2 degrees of freedom, truncated SE), inverse-variance Higgins I squared, Q and degrees of freedom. None if k<3. A hospital with any nan is dropped.
    hospitals with zero variance (v <= 1e-12) are dropped by default (exclude_zero_var=True). For such a hospital the retrieval prediction equals the frozen prediction, so the gain is also 0,
    and a variance floor of 1e-12 would give weight 1e12 and take over the whole pooled value (one hospital of F42 AKI retrieval held 99.9998% of the weight).
    exclude_zero_var=False keeps the variance floor 1e-12 in the weights (for the diagnostic columns *_orig and the re-aggregation of the reference hospital file a59).
    k_zero_var is the number of dropped hospitals, top1_weight_share is the largest share of weight held by one hospital."""
    d = np.asarray(d, dtype=float)
    v = np.asarray(v, dtype=float)
    zero = np.isfinite(d) & np.isfinite(v) & (v <= 1e-12)
    k_zero_var = int(zero.sum())
    ok = np.isfinite(d) & np.isfinite(v)
    if exclude_zero_var:
        ok &= ~zero
    d, v = d[ok], np.maximum(v[ok], 1e-12)
    k = len(d)
    if k < 3:
        return None
    w = 1.0 / v
    top1_weight_share = float(w.max() / w.sum())
    mu_fe = float((w * d).sum() / w.sum())
    Q = float((w * (d - mu_fe) ** 2).sum())
    I2 = max(0.0, (Q - (k - 1)) / Q) * 100 if Q > 0 else 0.0

    def nll(t2):
        wt = 1.0 / (v + t2)
        mu = (wt * d).sum() / wt.sum()
        return 0.5 * (np.log(v + t2).sum() + np.log(wt.sum()) + (wt * (d - mu) ** 2).sum())

    upper = max(10.0 * float(np.var(d, ddof=1)), 10.0 * float(v.max()), 1e-6)
    r = minimize_scalar(nll, bounds=(0.0, upper), method="bounded", options={"xatol": 1e-12})
    t2 = float(r.x) if nll(float(r.x)) < nll(0.0) - 1e-12 else 0.0
    wt = 1.0 / (v + t2)
    mu = float((wt * d).sum() / wt.sum())
    se_re = float(np.sqrt(1.0 / wt.sum()))
    q = float((wt * (d - mu) ** 2).sum() / (k - 1))
    se_hk = float(np.sqrt(q / wt.sum()))
    se_hkf = float(np.sqrt(max(1.0, q) / wt.sum()))
    tc = float(scipy_stats.t.ppf(0.975, k - 1))
    tp = float(scipy_stats.t.ppf(0.975, k - 2))
    s2 = float((k - 1) * w.sum() / (w.sum() ** 2 - (w ** 2).sum()))
    half_pi = tp * float(np.sqrt(t2 + se_hkf ** 2))
    return {"k": k, "mu_fixed": mu_fe, "Q": Q, "I2_invvar": I2, "tau2": t2, "tau": float(np.sqrt(t2)),
            "I2_reml": float(100 * t2 / (t2 + s2)) if (t2 + s2) > 0 else 0.0,
            "mu_random": mu, "se_random": se_re,
            "ci_z_lo": mu - 1.96 * se_re, "ci_z_hi": mu + 1.96 * se_re,
            "ci_hk_lo": mu - tc * se_hk, "ci_hk_hi": mu + tc * se_hk,
            "ci_hk_trunc_lo": mu - tc * se_hkf, "ci_hk_trunc_hi": mu + tc * se_hkf,
            "df": k - 1, "Q_p": float(scipy_stats.chi2.sf(Q, k - 1)), "q_hk": q,
            "pi_lo": mu - half_pi, "pi_hi": mu + half_pi,
            "k_zero_var": k_zero_var, "top1_weight_share": top1_weight_share}


def i2_stays_weighted(d, n):
    """Stay-weighted I squared (Q with stays as weights). The weight is not the inverse variance, so it is non-standard."""
    d = np.asarray(d, dtype=float)
    w = np.asarray(n, dtype=float)
    if len(d) < 3:
        return np.nan
    mu = float(np.average(d, weights=w))
    Q = float(((d - mu) ** 2 * w).sum())
    return max(0.0, (Q - (len(d) - 1)) / Q * 100) if Q > 0 else 0.0


# ---- actual MIMIC-IV year periods (A1) -----------------------------------------------
def read_mimic4_periods():
    """actual year period of a stay (MIMIC-IV). Two methods.
    (a) unit-transfer adjustment (M4_HAS_SHIFT): when patients has anchor_year and icustays has intime. anchor_year_group of patients is
        the actual 3-year period, and the period is shifted by the difference between the shifted year of the stay (intime) and anchor_year (PhysioNet MIMIC-IV documentation).
        if the stay falls entirely inside one 3-year period it is strict; if it straddles, it is dropped from strict. majority puts it in the period with most of its years.
    (b) patient-level approximation (copy without anchor_year): all stays of the same patient get the anchor_year_group period of that patient. There is no unit-transfer
        adjustment and intime is not used. MIMIC-IV shifts dates by a different amount per patient, so the intime order between patients is not the real calendar order.
        the patient_group column of the table is the period number (A1b uses only patient_group in this method)."""
    c_st = [_ci["stay_id"], _ci["subject_id"]] + ([_ci["intime"]] if M4_HAS_SHIFT else [])
    c_pt = [_cp["subject_id"], _cp["anchor_year_group"]] + ([_cp["anchor_year"]] if M4_HAS_SHIFT else [])
    # usecols returns columns in file order, so select again by name (naming by position would swap stay_id and subject_id)
    st = pd.read_csv(MIMIC4_ICUSTAYS_PATH, usecols=c_st)[c_st]
    st.columns = ["stay_id", "subject_id"] + (["intime"] if M4_HAS_SHIFT else [])
    pt = pd.read_csv(MIMIC4_PATIENTS_PATH, usecols=c_pt)[c_pt]
    pt.columns = ["subject_id", "anchor_year_group"] + (["anchor_year"] if M4_HAS_SHIFT else [])
    pt = pt.drop_duplicates("subject_id")
    m = st.merge(pt, on="subject_id", how="left")
    grp = m["anchor_year_group"].astype(str).str.extract(r"(\d{4})\s*-\s*(\d{4})")
    lo = pd.to_numeric(grp[0], errors="coerce")
    hi = pd.to_numeric(grp[1], errors="coerce")
    pairs = sorted({(int(a), int(b)) for a, b in zip(lo.dropna(), hi.dropna())})
    labels = [f"{a}-{b}" for a, b in pairs]
    if not M4_HAS_SHIFT:
        at = {p_: i_ for i_, p_ in enumerate(pairs)}
        pg = np.full(len(m), -1, dtype=int)
        for i, (a, b) in enumerate(zip(lo.to_numpy(dtype=float), hi.to_numpy(dtype=float))):
            if np.isfinite(a) and np.isfinite(b):
                pg[i] = at[(int(a), int(b))]
        out = pd.DataFrame({"stay_id": pd.to_numeric(m["stay_id"], errors="coerce").astype("int64"),
                            "shift": np.nan, "strict": pg, "majority": pg, "patient_group": pg})
        return out.set_index("stay_id"), labels
    shift = pd.to_datetime(m["intime"], errors="coerce").dt.year - pd.to_numeric(m["anchor_year"], errors="coerce")
    r_lo, r_hi = (lo + shift).to_numpy(dtype=float), (hi + shift).to_numpy(dtype=float)

    def which(y_):
        for gi_, (a, b) in enumerate(pairs):
            if a <= y_ <= b:
                return gi_
        return -1

    strict = np.full(len(m), -1, dtype=int)
    major = np.full(len(m), -1, dtype=int)
    for i in range(len(m)):
        if not (np.isfinite(r_lo[i]) and np.isfinite(r_hi[i])):
            continue
        yrs = range(int(r_lo[i]), int(r_hi[i]) + 1)
        gs = [which(y_) for y_ in yrs]
        if all(g_ == gs[0] for g_ in gs) and gs[0] >= 0:
            strict[i] = gs[0]
        cnt = {}
        for g_ in gs:
            if g_ >= 0:
                cnt[g_] = cnt.get(g_, 0) + 1
        if cnt:
            major[i] = sorted(cnt, key=lambda g_: (-cnt[g_], g_))[0]
    out = pd.DataFrame({"stay_id": pd.to_numeric(m["stay_id"], errors="coerce").astype("int64"),
                        "shift": shift.to_numpy(dtype=float), "strict": strict, "majority": major})
    return out.set_index("stay_id"), labels


# ---- temporal validation runner (A1) -------------------------------------------------------
def mean_abs_smd(XA, XB):
    """mean absolute standardized mean difference over columns of the standardized feature matrices of two periods."""
    ma, mb = XA.mean(0), XB.mean(0)
    sa, sb = XA.std(0, ddof=1), XB.std(0, ddof=1)
    sp = np.sqrt((sa ** 2 + sb ** 2) / 2.0)
    sp = np.where(sp < 1e-9, np.nan, sp)
    return float(np.nanmean(np.abs(ma - mb) / sp))


def run_temporal(C, past_mask, test_mask, name, outcome, banks, n_draws, groups):
    """splits the test-period patients into N_TIME_SPLITS folds stratified by outcome and tests fold by fold in turn (role alternation).
    P past bank (past period only) | S same-period bank (test period outside the test fold) | R (optional) everything outside the test fold (periods mixed).
    the three policies are scored on the same whole test patients, so differences between policies are paired. Compare at the same size only:
    if bank is "full", the policies are reduced to the largest size all can supply (n_max) and labeled "max"."""
    y = C["y"]
    test_ids = np.where(test_mask)[0]
    past_ids = np.where(past_mask)[0]
    notes = []
    if len(test_ids) < 4 * N_TIME_SPLITS or len(past_ids) < 50 or len(np.unique(y[test_ids])) < 2:
        notes.append({"analysis": name, "outcome": outcome, "note": f"insufficient sample (test period {len(test_ids)}, past {len(past_ids)})"})
        return [], [], notes
    folds = list(StratifiedKFold(n_splits=N_TIME_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(test_ids)), y[test_ids]))
    n_same = min(len(tr) for tr, _ in folds)                       # pool size of policy S (smallest fold)
    n_max = min(len(past_ids), n_same)
    rows, info_rows = [], []
    for bank_size in banks:
        size = n_max if bank_size == "full" else bank_size
        label = "max" if bank_size == "full" else str(bank_size)
        if size > n_max or size < 30:
            notes.append({"analysis": name, "outcome": outcome, "bank": label,
                          "note": f"insufficient pool (past {len(past_ids)} , same period {n_same} the smaller of {n_max} < bank {size})"})
            continue
        n_rep = 1 if bank_size == "full" else n_draws
        for pol in A1_POLICIES:
            for rep in range(n_rep):
                oof = {}
                for fi, (tr_i, te_i) in enumerate(folds):
                    test, other = test_ids[te_i], test_ids[tr_i]
                    pool = {"P": past_ids, "S": other, "R": np.concatenate([past_ids, other])}[pol]
                    key = f"a1|{name}|{pol}|{outcome}|{label}|{fi}|{rep}"
                    sel = np.arange(len(pool)) if size >= len(pool) else sample_bank(y[pool], size, key)
                    bank = pool[sel]
                    if np.intersect1d(bank, test).size or (pol == "P" and not past_mask[bank].all()):
                        raise AssertionError("invariant violation: the bank contains test patients or the P bank contains non-past patients " + key)
                    P, info = bank_predictions(C, bank, test, key, set(groups))
                    for m, v in P.items():
                        if m not in oof:
                            oof[m] = np.full(len(y), np.nan, dtype=np.float32)
                        oof[m][test] = v
                    info_rows.append({"analysis": name, "outcome": outcome, "bank": label, "policy": pol,
                                      "rep": rep, "fold": fi, "n_bank": int(len(bank)),
                                      "events_bank": int(y[bank].sum()), **info})
                for m, pm in oof.items():
                    pt = pm[test_ids].astype(float)
                    if np.isnan(pt).any():
                        continue
                    rows.append({"analysis": name, "outcome": outcome, "bank": label, "policy": pol,
                                 "rep": rep, "method": m, "n_test": int(len(test_ids)), "n_bank": int(size),
                                 **full_metrics(y[test_ids], pt)})
    return rows, info_rows, notes


def period_summary(C, past_mask, test_mask, name, outcome):
    """size, prevalence and feature shift of the past and test periods (for A1 description)."""
    y = C["y"]
    pa, te = np.where(past_mask)[0], np.where(test_mask)[0]
    smd = mean_abs_smd(C["Xg"][pa], C["Xg"][te]) if len(pa) > 2 and len(te) > 2 else np.nan
    return {"analysis": name, "outcome": outcome, "n_past": int(len(pa)), "n_test": int(len(te)),
            "prev_past": float(y[pa].mean()) if len(pa) else np.nan,
            "prev_test": float(y[te].mean()) if len(te) else np.nan, "mean_abs_smd": smd}


# ---- TabPFN connection (A9) ------------------------------------------------------------
TABPFN_INFO = {}                  # filled once when the first classifier is created: package version, v2 selection method, model_version, weight path argument


def tabpfn_classifier(device, seed):
    """creates the classifier by selecting the TabPFN v2 weights explicitly. Only the arguments accepted by the installed tabpfn are passed (the signature is read).
    behaviour by version (read from the sources of tabpfn 2.0.9, 2.1.3, 2.2.1, 6.0.0 and 9.1.0):
      - in 6.0.0 and 9.1.0, TabPFNClassifier.create_default_for_version(ModelVersion.V2, **kw) puts the v2 weight path in as model_path
        (classifier.py of 9.1.0 and 6.0.0. ModelVersion is in tabpfn.constants).
        versions between the two have not been tested. If the function is missing or fails, the fallback below is used and that fact is recorded in model_version.
        the default of 9.1.0 (when model_path is auto, settings.py, model_version=V3_5) is v3.5, not v2. So the default constructor is not used.
        the v2 weight file name has no v2.5, v2.6, v3 or v3.5 marker, so when 9.1.0 reads the version from the file name (model_loading.py) it reads V2.
      - 2.x has no selection function and only v2 weights, so the default constructor is v2 (ModelVersion in 2.2.1 model_loading.py has V2 only).
    if v2 cannot be selected, the run does not stop but continues with the default weights and records that fact in model_version."""
    from tabpfn import TabPFNClassifier
    params = inspect.signature(TabPFNClassifier.__init__).parameters
    kw = {}
    if "device" in params:
        kw["device"] = device
    if "n_estimators" in params:
        kw["n_estimators"] = TABPFN_N_ESTIMATORS
    if "random_state" in params:
        kw["random_state"] = seed
    factory = getattr(TabPFNClassifier, "create_default_for_version", None)
    clf, how, ver = None, "default constructor (this installation has no function to select the model version)", None
    if factory is not None:
        try:
            from tabpfn.constants import ModelVersion
            clf = factory(ModelVersion.V2, **kw)
            how, ver = "create_default_for_version(ModelVersion.V2)", "v2"
        except Exception as exc:  # noqa: BLE001
            how = f"v2 selection failed ({type(exc).__name__}: {str(exc)[:80]}), continuing with the default weights"
            ver = "unconfirmed: v2 selection failed, default weights"
    if clf is None:
        clf = TabPFNClassifier(**kw)
        if ver is None:
            ver = "v2 (tabpfn 2.x has v2 weights only)" if TABPFN_PKG_VERSION.startswith("2.") else "unconfirmed: no function to select the model version"
    if not TABPFN_INFO:
        TABPFN_INFO.update({"tabpfn_package_version": TABPFN_PKG_VERSION, "how_v2_selected": how, "model_version": ver,
                            "model_path_arg": str(getattr(clf, "model_path", "attribute absent")), "device": str(device),
                            "n_estimators": str(TABPFN_N_ESTIMATORS)})
    return clf


def tabpfn_info_rows():
    """installation and weight table printed by A9. Returns two sets (rows with full paths for the screen, rows with file names only for csv; the csv rows contain file names only, no folder paths)."""
    mp = TABPFN_INFO.get("model_path_arg", "")
    if mp in ("", "auto", "attribute absent"):
        wfile = "model_path is " + (mp if mp else "empty") + " (package default path, the file name is decided by this installation)"
    else:
        _f = Path(mp)
        wfile = _f.name + (" (present %.0f MB)" % (_f.stat().st_size / 1e6) if _f.is_file() else " (file not found)")
    items = [("tabpfn package version", TABPFN_INFO.get("tabpfn_package_version", "unknown")), ("v2 selection method", TABPFN_INFO.get("how_v2_selected", "unknown")),
             ("model_version", TABPFN_INFO.get("model_version", "unknown")), ("weight file", wfile),
             ("softmax_temperature (value after fitting)", TABPFN_INFO.get("softmax_temperature_after_fit", "before fitting")),
             ("device", TABPFN_INFO.get("device", "")), ("n_estimators", TABPFN_INFO.get("n_estimators", ""))]
    full = items + [("model_path argument (full path)", mp)]
    return full, items


def _is_oom(exc):
    """whether it is a GPU out-of-memory error (torch.OutOfMemoryError, TabPFNCUDAOutOfMemoryError, or out of memory in the message)."""
    return ("OutOfMemory" in type(exc).__name__) or ("out of memory" in str(exc).lower())


def tabpfn_predict(clf, X, batch=TABPFN_BATCH):
    """prediction in batches. On out of memory the batch is halved and the same batch is retried (minimum 8 rows). The predictions themselves do not depend on the batch size
    (TabPFN predicts each test row independently with the same context)."""
    out = np.empty(len(X), dtype=float)
    i, b = 0, max(int(batch), 1)
    while i < len(X):
        try:
            out[i:i + b] = clf.predict_proba(X[i:i + b])[:, 1]
            i += b
        except Exception as exc:  # noqa: BLE001
            if not _is_oom(exc) or b <= 8:
                raise
            b = max(b // 2, 8)
            try:
                import torch as _t
                _t.cuda.empty_cache()
            except Exception:  # noqa: BLE001
                pass
            print(f"  [A9] GPU out of memory , prediction batch reduced to {b} rows and retried", flush=True)
    return out


def tabpfn_fit_subsets(Xs, ys, n_ctx, n_sub, device, outcome):
    """fits TabPFN on n_sub stratified subsamples (n_ctx rows) of the development training data and freezes it (the model weights do not change)."""
    clfs = []
    for s in range(n_sub):
        idx = sample_bank(ys, n_ctx, f"tabpfn|{outcome}|{s}")
        clf = tabpfn_classifier(device, stable_seed("tabpfn", outcome, s))
        clf.fit(Xs[idx], ys[idx])
        TABPFN_INFO.setdefault("softmax_temperature_after_fit", str(getattr(clf, "softmax_temperature_", "attribute absent")))
        clfs.append(clf)
    return clfs


def tabpfn_frozen_probs(clfs, X):
    return np.mean([tabpfn_predict(c, X) for c in clfs], axis=0)


def stratified_cap(y, cap, key):
    """reduces the target cohort to cap rows (stratified). All rows if not reduced."""
    if cap is None or len(y) <= cap:
        return np.arange(len(y))
    return sample_bank(np.asarray(y), int(cap), key)


A9_RENAME = {"b1_base": "tabpfn_frozen", "b5_base": "logistic_frozen", "pram_b1": "pram_on_tabpfn",
             "pram_b5": "pram_on_logistic", "platt_b1": "platt_on_tabpfn", "recal_iso": "isotonic_on_tabpfn",
             "upd1_b1": "upd1_tabpfn", "upd2_b1": "upd2_tabpfn", "upd3_b1": "upd3_tabpfn", "ct_b1": "closed_test_tabpfn",
             "upd1_b5": "upd1_logistic", "upd2_b5": "upd2_logistic", "upd3_b5": "upd3_logistic", "ct_b5": "closed_test_logistic"}


# ---- reporting helpers (shared by the grid and hospital-level reporting cells) --------------------------------
def nbcol(t):
    return f"nb@{float(t)}"


def pick_report_banks(banks):
    pref = [b for b in (1000, "full") if b in banks]
    return pref if pref else [banks[0], banks[-1]]


def bank_sort_key(b):
    return 10 ** 9 if str(b) in ("full", "max") else int(b)


def draw_median_table(df, methods, extra_cols=()):
    """table of medians over draws (rep). Columns: AUROC, slope, intercept, ECE, net benefit at the representative threshold, net benefit at 20%."""
    d = df[df["method"].isin(methods)].copy()
    d["nb_site"] = [r[nbcol(T_SITE[oc])] for oc, r in zip(d["outcome"], d.to_dict("records"))]
    d["nb_20"] = d[nbcol(THRESH_MS)]
    cols = ["auroc", "slope", "intercept", "ece", "nb_site", "nb_20"] + list(extra_cols)
    g = d.groupby(["fs", "outcome", "target", "bank", "method"], sort=False)[cols].median().reset_index()
    g["n_draws"] = d.groupby(["fs", "outcome", "target", "bank", "method"], sort=False)["rep"].nunique().values
    return g


def paired_gap(df, method, base, col="auroc"):
    """summary of the distribution of the difference (col) between method and base in the same draws: median, 25%, 75%, minimum, fraction below 0."""
    keys = ["fs", "outcome", "target", "bank", "rep"]
    a = df[df["method"] == method].set_index(keys)[col]
    b = df[df["method"] == base].set_index(keys)[col]
    d = (a - b.reindex(a.index)).dropna()
    if len(d) == 0:
        return pd.DataFrame()
    g = d.groupby(["fs", "outcome", "target", "bank"], sort=False)
    out = g.agg(gap_median="median", gap_q25=lambda s: s.quantile(0.25), gap_q75=lambda s: s.quantile(0.75),
                gap_min="min", share_below_zero=lambda s: float((s < 0).mean()), n_draws="size").reset_index()
    out.insert(4, "method", method)
    out.insert(5, "base", base)
    return out


def hospital_time_unit(C, outcome, h, hid, year, y_past, y_test, groups):
    """A1 hospital level: the whole past year of hospital h is the bank and the whole next year is the test. Only hospitals above the criterion in both years."""
    y = C["y"]
    idx = np.where(hid == h)[0]
    bank, test = idx[year[idx] == y_past], idx[year[idx] == y_test]
    for part in (bank, test):
        ev = int(y[part].sum())
        if len(part) < HOSP_MIN_N_PER_YEAR or ev < HOSP_MIN_EVENTS_PER_YEAR or len(part) - ev < HOSP_MIN_EVENTS_PER_YEAR:
            return None
    P, info = bank_predictions(C, bank, test, f"a1h|{outcome}|{h}", set(groups))
    yt = y[test]
    rows = []
    for m, v in P.items():
        _a, _b, d1, v1 = delong_delta_var(yt, v, P["b1_base"])
        rows.append({"outcome": outcome, "hospitalid": int(h), "method": m, "n_bank": int(len(bank)), "n_test": int(len(test)),
                     "events_test": int(yt.sum()), **full_metrics(yt, v), "d_auroc_b1": d1, "var_b1": v1})
    return rows



def prepare_sites():
    """builds the hospital id array and the list of eligible hospitals (shared by the hospital-level cell and the network cell)."""
    hid, sites = {}, {}
    for oc in OUTCOMES:
        c = CTX[("F42", oc, "eICU")]
        hid[oc] = map_to_stays(EICU_TAB, "hospitalid", c["stay_ids"])
        s = eligible_sites(c["y"], hid[oc])
        sites[oc] = s[:SITE_LIMIT] if SITE_LIMIT else s
    return hid, sites

def as_table_frame(df, a="analysis", p="policy"):
    """temporarily renames the A1 result to the column names (fs, target) that draw_median_table and paired_gap read."""
    return df.rename(columns={a: "fs", p: "target"})


def policy_contrast(tab, method, base, col):
    """A1: puts the gains (col) of the same strategy side by side under policies P (past), S (same period) and R (if present), and gives P-S and P-R."""
    g = paired_gap(tab, method, base, col)
    if len(g) == 0:
        return pd.DataFrame()
    w = g.pivot_table(index=["fs", "outcome", "bank"], columns="target", values="gap_median", aggfunc="first")
    for pol in ("P", "S", "R"):
        if pol not in w.columns:
            w[pol] = np.nan
    w["P_minus_S"] = w["P"] - w["S"]
    w["P_minus_R"] = w["P"] - w["R"]
    w = w.reset_index().rename(columns={"fs": "analysis"})
    w.insert(3, "method", method)
    w.insert(4, "metric", col)
    return w


# ---- tail risk (A2): unit of (one bank, one test fold) -------------------------------------
def run_tail_unit(C, fs, outcome, target, bank_size, groups, keep, delong_pairs):
    """one cell (feature set, outcome, target, bank size). Makes N_FOLDS x N_DRAWS_TAIL // N_FOLDS (bank, test fold) pairs.
    one pair is one measurement in which one hospital receives one bank and measures on its own test patients."""
    y = C["y"]
    folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))
    per_fold = max(1, N_DRAWS_TAIL // N_FOLDS)
    rows = []
    for fi, (pool_idx, test_idx) in enumerate(folds):
        if bank_size > len(pool_idx):
            continue
        yt = y[test_idx]
        for dr in range(per_fold):
            key = f"{outcome}|{target}|{bank_size}|{fi}|{dr}"
            bank = pool_idx[sample_bank(y[pool_idx], bank_size, "tail|" + key)]
            if len(np.unique(y[bank])) < 2:
                continue
            P, _info = bank_predictions(C, bank, test_idx, key, set(groups))
            for m in keep:
                if m not in P:
                    continue
                row = {"fs": fs, "outcome": outcome, "target": target, "bank": str(bank_size), "fold": fi, "draw": dr,
                       "method": m, "n_events_bank": int(y[bank].sum()), **full_metrics(yt, P[m])}
                if delong_pairs.get(m) in P:
                    _a, _b, dd, vv = delong_delta_var(yt, P[m], P[delong_pairs[m]])
                    row["d_auroc_base"], row["var_auroc_base"] = dd, vv
                rows.append(row)
    return rows


def tail_gaps(df, pairs):
    """turns the tail rows (metrics per method) into a table of (method - reference) pair differences. Columns: gap_auroc gap_ece gap_slope gap_nb_site gap_nb_20 dl_lo dl_hi."""
    t = df.copy()
    t["nb_site"] = [r[nbcol(T_SITE[oc])] for oc, r in zip(t["outcome"], t.to_dict("records"))]
    t["nb_20"] = t[nbcol(THRESH_MS)]
    keys = ["fs", "outcome", "target", "bank", "fold", "draw"]
    cols = ["auroc", "ece", "slope", "nb_site", "nb_20"]
    out = []
    for m, b in pairs.items():
        a = t[t["method"] == m].set_index(keys)
        bb = t[t["method"] == b].set_index(keys)
        if len(a) == 0 or len(bb) == 0:
            continue
        g = (a[cols] - bb[cols].reindex(a.index)).add_prefix("gap_")
        g["method"] = m
        if "d_auroc_base" in a.columns:
            sd = np.sqrt(a["var_auroc_base"].astype(float))
            g["dl_lo"], g["dl_hi"] = a["d_auroc_base"] - 1.96 * sd, a["d_auroc_base"] + 1.96 * sd
        out.append(g.reset_index())
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def tail_summary(G):
    """tail of the gain distribution per (feature set, source, outcome, target, bank, method) cell. Cells without a metric (such as net benefit in the rows read from the reference file) are left empty."""
    rows = []
    for (fs, src, oc, tg, bk, m), s in G.groupby(["fs", "source", "outcome", "target", "bank", "method"], sort=False):
        a = s["gap_auroc"].dropna().to_numpy()
        if len(a) == 0:
            continue
        n = len(a)
        row = {"fs": fs, "source": src, "outcome": oc, "target": tg, "bank": bk, "method": m, "n_draws": n,
               "dAUROC_median": float(np.median(a)), "dAUROC_p05": float(np.quantile(a, 0.05)),
               "dAUROC_p10": float(np.quantile(a, 0.10)), "dAUROC_worst": float(a.min()), "share_below_zero": float((a < 0).mean())}
        for mg in NI_MARGINS_AUROC:
            row[f"auroc_share_lost_more_than_{mg}"] = float((a < -mg).mean())
        k = int((a < -NI_MARGIN_AUROC).sum())
        row["auroc_ni_share"] = k / n
        row["auroc_ni_cp_lo"], row["auroc_ni_cp_hi"] = cp_interval(k, n)
        if "dl_hi" in s.columns and s["dl_hi"].notna().any():
            dl = s.dropna(subset=["dl_lo", "dl_hi"])
            row["draws_ci_wholly_below_zero"] = float((dl["dl_hi"] < 0).mean())
            row["draws_ci_wholly_above_zero"] = float((dl["dl_lo"] > 0).mean())
        for col, tag in (("gap_ece", "ece"), ("gap_slope", "slope")):
            v = s[col].dropna()
            if len(v):
                row[f"d_{tag}_median"] = float(v.median())
        v = s["gap_ece"].dropna()
        if len(v):
            row["share_ece_worse"] = float((v > 0).mean())
        for col, tag in (("gap_nb_site", "nb_site"), ("gap_nb_20", "nb_20")):
            v = s[col].dropna().to_numpy()
            if len(v) == 0:
                continue
            row[f"d_{tag}_median"], row[f"d_{tag}_p05"], row[f"d_{tag}_worst"] = float(np.median(v)), float(np.quantile(v, 0.05)), float(v.min())
            for mg in NI_MARGINS_NB:
                row[f"{tag}_share_lost_more_than_{mg}"] = float((v < -mg).mean())
            kn = int((v < -NI_MARGIN_NB).sum())
            row[f"{tag}_ni_share"] = kn / len(v)
            row[f"{tag}_ni_cp_lo"], row[f"{tag}_ni_cp_hi"] = cp_interval(kn, len(v))
        rows.append(row)
    return pd.DataFrame(rows)


## Write lock and state check of notebook 01 (done before reading the data)

**Why**: if this notebook is not opened in the folder in which notebook 01 finished, a computation of 12 to 16 hours could start unnoticed. So two things are done before the data are read.

1. **Write lock.** `Resumable.add_row` and `mark_unit` write only to names in `CT_NEW_NAMES`. An attempt to write another name (an existing resume file) stops with `CtpubStop`. `save_csv` writes only names starting with `CTPUB_` (the saving of `frozen_model_auroc.csv` called by the data loading cell is also skipped).
2. **State check of notebook 01.** It counts whether the number of finished units of the grid resume (`matrix_F42`, `matrix_F33`) equals what the settings require and whether the step records of the variant (`matrixinfo_*`) exist. If they are insufficient, it stops without new computation.


In [ ]:
# ================================================================
# PRAM closed testing  write lock | state check of the stored results
# ================================================================
class CtpubStop(BaseException):
    """The stored results is not in the expected state or an attempt was made to write an existing file. Stops before computing. It is a BaseException, so `except Exception` does not swallow it."""


def ct_count_units(name):
    path = RESUME_DIR / f"{name}.units.jsonl"
    if not path.is_file():
        return 0
    with open(path, "r", encoding="utf-8") as _f:
        return sum(1 for _ln in _f if _ln.strip())


if not hasattr(Resumable, "_ct_add_row_plain"):
    Resumable._ct_add_row_plain = Resumable.add_row
    Resumable._ct_mark_unit_plain = Resumable.mark_unit

    def _ct_guarded_add_row(self, row):
        if self.name not in CT_NEW_NAMES:
            raise CtpubStop("stop , tried to write the existing resume file `%s`. This notebook writes only the new name %s. The existing file was not changed." % (self.name, list(CT_NEW_NAMES)))
        return Resumable._ct_add_row_plain(self, row)

    def _ct_guarded_mark_unit(self, unit):
        if self.name not in CT_NEW_NAMES:
            raise CtpubStop("stop , tried to write unit completion to the existing resume file `%s`. This notebook writes only the new name %s. The existing file was not changed." % (self.name, list(CT_NEW_NAMES)))
        return Resumable._ct_mark_unit_plain(self, unit)
    Resumable.add_row = _ct_guarded_add_row
    Resumable.mark_unit = _ct_guarded_mark_unit

_save_csv_plain = save_csv


def save_csv(df, name):
    """This notebook writes only names starting with CTPUB_. Other names (existing csv files of the stored results) are not written and one line is left."""
    if not name.startswith(CT_CSV_PREFIX):
        print("  (not saved) %s , not a new file of this notebook. The existing file is not touched" % name)
        return None
    return _save_csv_plain(df, name)


CT_EXP_UNITS = {_f: len(OUTCOMES) * len(FS_SPEC[_f]["targets"]) * len(FS_SPEC[_f]["banks"]) for _f in CT_FS}
CT_HAVE_UNITS = {_f: (ct_count_units(f"matrix_{_f}"), ct_count_units(f"matrixinfo_{_f}")) for _f in CT_FS}
CT_BAD_UNITS = {_f: v for _f, v in CT_HAVE_UNITS.items() if min(v) < CT_EXP_UNITS[_f]}
print("state check of the stored results , folder %s" % RESUME_DIR.resolve())
print("| feature set | grid finished units | stage record finished units | needed |")
print("|---|---|---|---|")
for _f in CT_FS:
    print("| %s | %d | %d | %d |" % (_f, CT_HAVE_UNITS[_f][0], CT_HAVE_UNITS[_f][1], CT_EXP_UNITS[_f]))
if CT_BAD_UNITS:
    raise CtpubStop("stop , the grid results of the stored results are insufficient (feature set: (grid finished units, stage record finished units)) %s , needed %s . This folder (%s) is not the folder of the stored results or the stored results did not finish. "
                    "No new computation was started. Open this notebook in the same folder as the stored results (where out_pram is)."
                    % (CT_BAD_UNITS, {_f: CT_EXP_UNITS[_f] for _f in CT_BAD_UNITS}, RESUME_DIR.resolve()))
print("state of the stored results confirmed , write lock on (only %d new names are written)" % len(CT_NEW_NAMES))


## Data loading, feature sets, frozen models

The six input files described in the repository README are read from `prepared_data/`. The 33-column set drops the variables whose maximum missingness over the three databases exceeds 30% (the 9 columns are removed before input and the development cache is rebuilt). The notebook prints whether the dropped variables are the nine expected ones (temperature, INR and weight; minimum, maximum and last).

For each feature set the development models (frozen logistic regression and CatBoost) are refitted on the MIMIC-IV training 56%.

| Set | Columns | Targets | Use |
|---|---|---|---|
| F42 | 42 inputs (43 after transform) | three cohorts | main analysis and update hierarchy; reference check |
| F33 | 33 inputs (34 after transform) | three cohorts | 33-column analyses |
| F42_noTemp, F42_noINR | only the three temperature columns or only the three INR columns removed | eICU | off (runs if added to `FS_RUN`) |

The printout of the number of inputs and of columns after the transform (42, 43, 33 and 34) is the feature count check. The column names are listed in Supplementary S1 Panel B.


In [ ]:
# ================================================================
# PRAM Cell 4. Data loading, feature sets, frozen models, A10
# ================================================================
DS, DEV, CTX, FS_COLS = {}, {}, {}, {}
EICU_TAB, SITE_CODE = None, {}
CAT_REF, NUM_REF, ALLVARS, SPARSE = [], [], [], []
MISS_TAB = pd.DataFrame()
STATUS = {}                                   # status per section. Read by the closing table (was it finished) of the last cell


def status(section, state, note=""):
    STATUS[section] = (state, note)


if not DATA_OK:
    emit("**[skipped] The main inputs are missing.** Look at the input check table of cell 1, fix `DATA_ROOT` and run again. All analysis cells below are skipped.")
    status("data loading", "not available", "main inputs absent")
else:
    t_load = time.time()
    for _oc in OUTCOMES:
        DS[("MIMIC-IV", _oc)] = load_mimic4(_oc)
        DS[("MIMIC-III", _oc)] = load_external("mimic3", _oc)
        DS[("eICU", _oc)] = load_external("eicu", _oc)
    X_REF, _ = _load_x("mimic4")
    CAT_REF, NUM_REF = _detect_cols(X_REF)
    ALLVARS = CAT_REF + NUM_REF
    _rows = []
    for _name in TARGETS:
        _Xall = X_REF if _name == "MIMIC-IV" else DS[(_name, "AKI_168h")]["X_df"]
        for _c in ALLVARS:
            _rows.append({"database": _name, "variable": _c, "pct_missing": 100.0 * float(_Xall[_c].isna().mean())})
    MISS_TAB = pd.DataFrame(_rows)
    _mm = MISS_TAB.groupby("variable")["pct_missing"].max()
    SPARSE = sorted([c for c in ALLVARS if _mm[c] > MISS_CUT_PCT])
    emit("### Cohort")
    _coh = [{"database": n, "outcome": oc, "n": len(DS[(n, oc)]["y"]), "prevalence": float(DS[(n, oc)]["y"].mean()),
             "n_inputs": DS[(n, oc)]["X_df"].shape[1]} for n in TARGETS for oc in OUTCOMES]
    emit_table("cohort size and prevalence", pd.DataFrame(_coh))
    emit(f"- number of feature inputs {len(ALLVARS)} (expected 42) | categorical {CAT_REF} | continuous {len(NUM_REF)} columns")
    emit(f"- maximum missingness over the three databases is {MISS_CUT_PCT}% are dropped {len(SPARSE)} columns: {SPARSE}")
    emit("- the nine expected variables and " + ("**equal**" if SPARSE == sorted(EXPECTED_SPARSE_9) else f"**differ** (expected {sorted(EXPECTED_SPARSE_9)})"))
    for _fs in FS_RUN:
        _sp = FS_SPEC[_fs]
        _cols = [c for c in ALLVARS if not (_sp["drop_rule"] == "miss" and c in SPARSE)
                 and not any(c.startswith(p) for p in _sp["drop_prefix"])]
        FS_COLS[_fs] = _cols
    emit("- number of feature-set columns: " + ", ".join(f"{k} {len(v)}" for k, v in FS_COLS.items()))
    status("data loading", "done", f"{time.time() - t_load:.0f} s")

    t_dev = time.time()
    for _fs in FS_RUN:
        for _oc in OUTCOMES:
            _d4 = DS[("MIMIC-IV", _oc)]
            _tr = _d4["idx_train"]
            DEV[(_fs, _oc)] = build_dev(_d4["X_df"].iloc[_tr], _d4["y"][_tr], FS_COLS[_fs], CAT_REF, NUM_REF, f"{_fs}|{_oc}")
            for _tg in FS_SPEC[_fs]["targets"]:
                _d = DS[(_tg, _oc)]
                if _tg == "MIMIC-IV":
                    _te = _d["idx_test"]
                    CTX[(_fs, _oc, _tg)] = make_ctx(DEV[(_fs, _oc)], _d["X_df"].iloc[_te], _d["y"][_te], _d["stay_ids"][_te], _oc)
                else:
                    CTX[(_fs, _oc, _tg)] = make_ctx(DEV[(_fs, _oc)], _d["X_df"], _d["y"], _d["stay_ids"], _oc)
            _log(f"frozen model fit {_fs} {_oc}")
    _chk = []
    for (_fs, _oc, _tg), _c in CTX.items():
        assert 0.0 <= _c["pb1"].min() and _c["pb1"].max() <= 1.0 and 0.0 <= _c["pb5"].min() and _c["pb5"].max() <= 1.0, "the frozen model output is not a probability"
        assert len(_c["y"]) == len(_c["Xg"]) == len(_c["Xc"]) == len(_c["stay_ids"]), "the context array lengths do not match"
        _a1 = roc_auc_score(_c["y"], _c["pb1"])
        _a5 = roc_auc_score(_c["y"], _c["pb5"])
        _chk.append({"fs": _fs, "outcome": _oc, "target": _tg, "n": len(_c["y"]), "prev": float(_c["y"].mean()),
                     "d": _c["d"], "auroc_logistic": _a1, "auroc_catboost": _a5})
    EICU_TAB = load_eicu_patient_table() if (HAVE_HOSPITALID or HAVE_EICU_YEAR) else None
    if EICU_TAB is not None and "hospitalid" in EICU_TAB.columns:        # only this code, not the hospital id, goes to the result csv
        SITE_CODE = {int(h): f"H{k + 1:03d}" for k, h in enumerate(sorted(EICU_TAB["hospitalid"].dropna().unique()))}
    emit_table("frozen model performance", pd.DataFrame(_chk), max_rows=60)
    save_csv(pd.DataFrame(_chk), "frozen_model_auroc")
    status("frozen model", "done", f"{time.time() - t_dev:.0f} s")
    for _fs in [f for f in ("F42", "F33") if f in FS_COLS]:                  # feature count consistency: number of inputs and number of columns after the transform
        _pg = DEV[(_fs, "AKI_168h")]["pg"]
        emit(f"- feature count consistency {_fs}: input {len(FS_COLS[_fs])} -> model columns after the transform {len(_pg.column_names())} (categorical {len(_pg.cat_cols)} expanded as one-hot)")


## Implementation of the published procedure (new definitions)

The definitions below transfer steps B to D of Vergouwe et al. (2017) and the appendix R code (`ClosedTest`). The variant function `update_hierarchy` of notebook 01 is not changed.

| Published procedure | This notebook |
|---|---|
| model 0 original model, 1 intercept, 2 intercept and slope, 3 revision (Table I) | `ll0`, `fit_intercept_only`, `_newton_2`, `fit_revision_mle` |
| step B: revision versus original, p + 1 degrees of freedom, original if not significant | `p_B`, degrees of freedom `rank` |
| step C: revision versus intercept update, p degrees of freedom, intercept if not significant | `p_C`, degrees of freedom `rank - 1` |
| step D: revision versus recalibration, p - 1 degrees of freedom, recalibration if not significant, revision if significant | `p_D`, degrees of freedom `rank - 2` |
| revision = unpenalized re-estimation of all coefficients (`lrm.fit(x = X, y = y)`) | `fit_revision_mle` (IRLS, no penalty, iteration cap 25 and convergence 1e-8 as in R `glm`) |
| shrinkage is not part of the procedure (page 4536) | if the revision is chosen, the prediction of the unpenalized revision is used as it is |

**Degrees of freedom and design rank**: p is the number of regression coefficients of the original model (excluding the intercept); in this notebook it is the number of model columns after the transform (`Xb.shape[1]`). If a categorical column is all zero in the bank (constant after standardization), the rank of the design matrix `[1, X]` is below `p + 1`. In that case the degrees of freedom are counted with the number of identifiable directions `rank` (if `rank = p + 1` they are p + 1, p and p - 1 as published). The fraction of folds with deficient rank appears in `CTPUB_step_shares` as `share_rank_deficient`.

**Difference from R `lrm.fit`**: the R code of the published procedure fits the revision with `lrm.fit`. Its convergence criterion is the change of -2 log likelihood (`eps`); the default of the current rms documentation is `maxit = 50` and `eps = 5e-4`. The 2013 default (the date at the head of the appendix R code) is not documented here. The iteration here continues to a relative deviance change of 1e-8, so it is tighter. In folds without separation both criteria reach the same maximum likelihood and the likelihood-ratio statistic differs only on the scale of `eps`; in folds with separation the values depend on the iteration cap (below).

**Folds in which the unpenalized revision does not exist**: in a bank with few events the maximum likelihood may not exist (complete separation). With the same criterion as R `glm` (fitted probability within `10 x machine epsilon` of 0 or 1), `separated` is marked, and the test and prediction are given with the value at the iteration cap (25). The published procedure does not define this case. The fraction appears as `share_separated`.

**Four diagnostics given together** (to separate why steps differ): in the same fold, `step_pub` (published procedure, unpenalized maximum likelihood), `step_pub_ridge` (the test order of the published procedure with the likelihood of the penalized revision of the variant), `step_var_exact` (the test order of the variant with the unpenalized maximum likelihood), `step_var_ridge` (the variant recomputed; must equal `ct_step_b1` stored by the grid of notebook 01). The difference between the first two is the effect of the **test order**; the difference between the last two is the effect of the **revision fit (tiny penalty)**.


In [ ]:
# ================================================================
# PRAM closed testing  implementation of the published procedure (Vergouwe 2017 steps B-D, appendix R code)
# ================================================================
def fit_revision_mle(Xb, yb, offb, maxit=CT_MLE_MAXIT, eps=CT_MLE_EPS):
    """unpenalized maximum likelihood of the model revision: solves z = off + a + X u by IRLS (Newton, halving the step if the deviance does not decrease). Corresponds to lrm.fit(x = X, y = y) of the published procedure.
    uses only the identifiable directions (rank r) from the singular value decomposition of the design matrix [1, X]. The convergence criterion and iteration cap are the R glm defaults (relative deviance change 1e-8, 25 iterations).
    returns: a, u, log likelihood, rank r, convergence, number of iterations, complete separation (fitted probability within 10 x epsilon of 0 or 1)."""
    X = np.asarray(Xb, dtype=float)
    y = np.asarray(yb, dtype=float)
    off = np.asarray(offb, dtype=float)
    Z = np.column_stack([np.ones(len(y)), X])
    _U, s, Vt = np.linalg.svd(Z, full_matrices=False)
    r = int((s > s[0] * CT_RANK_TOL).sum())
    V = Vt[:r].T
    Zr = Z @ V
    gam = np.zeros(r)
    z = off.copy()
    dev = 2.0 * _nll_sum(y, z)
    converged, n_it = False, 0
    for n_it in range(1, maxit + 1):
        pr = _sigmoid(z)
        w = pr * (1.0 - pr) + 1e-12
        grad = Zr.T @ (y - pr)
        H = (Zr * w[:, None]).T @ Zr
        try:
            step = np.linalg.solve(H, grad)
        except np.linalg.LinAlgError:
            step = np.linalg.lstsq(H, grad, rcond=None)[0]
        t = 1.0
        while True:
            z_new = off + Zr @ (gam + t * step)
            dev_new = 2.0 * _nll_sum(y, z_new)
            if dev_new <= dev + 1e-12 * (abs(dev) + 1.0) or t < 1e-6:
                break
            t *= 0.5
        dev_old, gam, z, dev = dev, gam + t * step, z_new, dev_new
        if abs(dev_old - dev) / (abs(dev) + 0.1) < eps:
            converged = True
            break
    theta = V @ gam
    pr = _sigmoid(z)
    separated = bool((pr > 1.0 - CT_SEP_EPS).any() or (pr < CT_SEP_EPS).any())
    return {"a": float(theta[0]), "u": theta[1:].copy(), "ll": -dev / 2.0, "rank": r, "converged": converged, "iters": n_it, "separated": separated}


def ct_pub_step(p_b, p_c, p_d):
    """decision of the published procedure: 0 if B is not significant, 1 if C is not significant, 2 if D is not significant, otherwise 3."""
    if p_b >= CT_ALPHA:
        return 0
    if p_c >= CT_ALPHA:
        return 1
    if p_d >= CT_ALPHA:
        return 2
    return 3


def ct_var_step(p_all, p_rev_vs_recal, p_slope):
    """decision of the variant (same rule as update_hierarchy): 0 if revision versus no update is not significant, 3 if revision versus recalibration is significant, 2 if the slope test is significant, otherwise 1."""
    if p_all >= CT_ALPHA:
        return 0
    if p_rev_vs_recal < CT_ALPHA:
        return 3
    if p_slope < CT_ALPHA:
        return 2
    return 1


def _chi2_p(lr, df):
    return float(scipy_stats.chi2.sf(max(0.0, lr), max(int(df), 1)))


def closed_test_published(Xb, yb, offb, Xt, offt, p_frozen_t):
    """fits update steps 0 to 3 in the bank (Xb, yb, offb = frozen linear predictor), chooses one step by the published closed testing (B, C, D), then gives predictions on the test data (Xt, offt).
    the revision is the unpenalized maximum likelihood (no shrinkage). Diagnostics: step_pub_ridge, step_var_exact, step_var_ridge (see the function header).
    if the number of events is below N_EVENT_MIN, the linear predictor is not finite or the slope fit fails, returns step 0 (frozen prediction) with ok=False, as update_hierarchy does."""
    res = {"pub": p_frozen_t, "step": 0, "ok": False, "step_pub_ridge": 0, "step_var_exact": 0, "step_var_ridge": 0, "rank": np.nan,
           "d": int(Xb.shape[1]), "converged": False, "iters": 0, "separated": False,
           "p_B": np.nan, "p_C": np.nan, "p_D": np.nan, "lr_B": np.nan, "lr_C": np.nan, "lr_D": np.nan, "dll3": np.nan}
    yb = np.asarray(yb).astype(float)
    n1 = int(yb.sum())
    n0 = int(len(yb) - n1)
    if min(n1, n0) < N_EVENT_MIN or not np.isfinite(offb).all() or not np.isfinite(offt).all():
        return res
    d = res["d"]
    ll0 = -_nll_sum(yb, offb)
    a1 = fit_intercept_only(yb, offb)
    ll1 = -_nll_sum(yb, offb + a1)
    a2, b2 = _newton_2(yb, offb)
    if not np.isfinite(a2):
        return res
    ll2 = -_nll_sum(yb, a2 + b2 * offb)
    a3r, u3r, _ok = fit_revision(Xb, yb, offb)
    ll3r = -_nll_sum(yb, offb + a3r + Xb @ u3r)
    fit = fit_revision_mle(Xb, yb, offb)
    ll3m, r = fit["ll"], fit["rank"]
    lr_b, lr_c, lr_d = 2 * (ll3m - ll0), 2 * (ll3m - ll1), 2 * (ll3m - ll2)
    p_b, p_c, p_d = _chi2_p(lr_b, r), _chi2_p(lr_c, r - 1), _chi2_p(lr_d, r - 2)
    step = ct_pub_step(p_b, p_c, p_d)
    p_slope = _chi2_p(2 * (ll2 - ll1), 1)
    res.update({"step": step, "ok": True, "rank": r, "converged": fit["converged"], "iters": fit["iters"], "separated": fit["separated"],
                "p_B": p_b, "p_C": p_c, "p_D": p_d, "lr_B": max(0.0, lr_b), "lr_C": max(0.0, lr_c), "lr_D": max(0.0, lr_d), "dll3": float(ll3m - ll3r)})
    res["step_pub_ridge"] = ct_pub_step(_chi2_p(2 * (ll3r - ll0), d + 1), _chi2_p(2 * (ll3r - ll1), d), _chi2_p(2 * (ll3r - ll2), d - 1))
    res["step_var_ridge"] = ct_var_step(_chi2_p(2 * (ll3r - ll0), d + 1), _chi2_p(2 * (ll3r - ll2), max(d - 1, 1)), p_slope)
    res["step_var_exact"] = ct_var_step(p_b, _chi2_p(lr_d, r - 2), p_slope)
    pred = [p_frozen_t, _sigmoid(offt + a1), _sigmoid(a2 + b2 * offt), _sigmoid(offt + fit["a"] + Xt @ fit["u"])]
    res["pub"] = pred[step]
    return res


CT_FOLD_FIELDS = ("step", "step_pub_ridge", "step_var_exact", "step_var_ridge", "rank", "d", "converged", "iters", "separated",
                  "p_B", "p_C", "p_D", "lr_B", "lr_C", "lr_D", "dll3", "ok")


def ct_fold_row(res):
    """the diagnostics of one fold as resume rows (predictions are left out). step is the step of the published procedure."""
    row = {k: res[k] for k in CT_FOLD_FIELDS}
    row["step_pub"] = row.pop("step")
    return row


def run_ctpub_unit(C, fs, outcome, target, bank_size, n_draws):
    """one unit (feature set, outcome, target, bank size). The split, draw key (m23) and bank are the same as in run_matrix_unit (the same patients are drawn).
    collects the 5-fold test predictions per draw (rep) and gives the metrics of the published procedure (ct_pub). Returns: (metric rows per draw, diagnostic rows per fold)."""
    y = C["y"]
    n = len(y)
    folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(n), y))
    n_rep = 1 if bank_size == "full" else n_draws
    rows, fold_rows = [], []
    for rep in range(n_rep):
        oof = np.full(n, np.nan, dtype=np.float32)
        for fi, (pool_idx, test_idx) in enumerate(folds):
            key = f"{outcome}|{target}|{bank_size}|{fi}|{rep}"
            sel = np.arange(len(pool_idx)) if bank_size == "full" else sample_bank(y[pool_idx], bank_size, "m23|" + key)
            bank = pool_idx[sel]
            p1b, p1t = C["pb1"][bank], C["pb1"][test_idx]
            res = closed_test_published(C["Xg"][bank], y[bank], _logit(p1b), C["Xg"][test_idx], _logit(p1t), p1t)
            oof[test_idx] = res["pub"]
            fold_rows.append({"fs": fs, "outcome": outcome, "target": target, "bank": str(bank_size), "rep": rep, "fold": fi,
                              "n_bank": int(len(bank)), "events_bank": int(y[bank].sum()), **ct_fold_row(res)})
        if not np.isnan(oof).any():
            rows.append({"fs": fs, "outcome": outcome, "target": target, "bank": str(bank_size), "rep": rep, "method": "ct_pub",
                         **full_metrics(y, oof.astype(float))})
    return rows, fold_rows


def ctpub_hospital_unit(C, outcome, h, hid):
    """one hospital (3-fold OOF). The fold split and bank are the same as in hospital_oof (key a29). Returns: (metric rows, diagnostic rows per fold). (None, None) if not eligible."""
    y = C["y"]
    idx = np.where(hid == h)[0]
    ys = y[idx]
    n_sp = min(N_SITE_SPLITS, int(ys.sum()), int(len(ys) - ys.sum()))
    if n_sp < 2:
        return None, None
    skf = StratifiedKFold(n_splits=n_sp, shuffle=True, random_state=stable_seed("a29", outcome, int(h)))
    oof = np.full(len(ys), np.nan)
    fold_rows = []
    for fi, (bank_i, test_i) in enumerate(skf.split(np.zeros(len(ys)), ys)):
        bank, test = idx[bank_i], idx[test_i]
        yb, yte = y[bank], y[test]
        if len(np.unique(yb)) < 2 or len(np.unique(yte)) < 2:
            return None, None
        p1b, p1t = C["pb1"][bank], C["pb1"][test]
        res = closed_test_published(C["Xg"][bank], yb, _logit(p1b), C["Xg"][test], _logit(p1t), p1t)
        oof[test_i] = res["pub"]
        fold_rows.append({"outcome": outcome, "hospitalid": int(h), "fold": fi, "n_test": int(len(test)), "events_test": int(yte.sum()),
                          "n_bank": int(len(bank)), "events_bank": int(yb.sum()), **ct_fold_row(res)})
    if np.isnan(oof).any():
        return None, None
    base1 = C["pb1"][idx]
    _a, _b, d1, v1 = delong_delta_var(ys, oof, base1)
    row = {"outcome": outcome, "hospitalid": int(h), "method": "ct_pub", "n_site": int(len(ys)), "n_events": int(ys.sum()), "prev": float(ys.mean()),
           **full_metrics(ys, oof), "d_auroc_b1": d1, "var_b1": v1}
    return [row], fold_rows


## Pre-check: finished units, units to run, expected time

This cell **does not do new computation** (only one fold is fitted at a time to measure the expected time; nothing is stored). It prints the following.

1. Number of units to run and units to skip (finished units of this notebook in the folder are skipped).
2. Number of fold fits and an expected time (approximate) from the measured time of one fold per bank size.
3. Hospital level: whether the number of hospital units finished by notebook 01 equals the number required now (if different, it stops before computing).

**No predictions are stored.** The resume of notebook 01 holds only metrics and step records, no predictions. So bank fits are redone, but the same split and the same draw key (`m23`) make the same banks, and the values of the variant, retrieval and the frozen models are read from the stored metrics.


In [ ]:
# ================================================================
# PRAM closed testing  pre-check (new units | expected time | number of hospital units)
# ================================================================
CT_PLAN = pd.DataFrame()
CT_HID, CT_SITES = {}, {}
CT_RUN_HOSP = False


def ct_time_one_fold(C, bank_size, key):
    """fit time (seconds) of the published procedure for one fold. For the time estimate; results are not stored."""
    y = C["y"]
    folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))
    pool_idx, test_idx = folds[0]
    sel = np.arange(len(pool_idx)) if bank_size == "full" else sample_bank(y[pool_idx], bank_size, "ctplan|" + key)
    bank = pool_idx[sel]
    p1b, p1t = C["pb1"][bank], C["pb1"][test_idx]
    t0 = time.time()
    closed_test_published(C["Xg"][bank], y[bank], _logit(p1b), C["Xg"][test_idx], _logit(p1t), p1t)
    return time.time() - t0, len(pool_idx)


if not DATA_OK:
    status("closed-testing pre-check", "not available", "main inputs absent")
else:
    _plan_rows, _sec_total = [], 0.0
    _t_ref = {}
    for _f in CT_FS:
        _have_methods = set(Resumable(f"matrix_{_f}").df()["method"])
        _need_methods = {"b1_base", "pram_b1", "ct_b1", "upd1_b1", "upd2_b1", "upd3_b1"}
        if not _need_methods <= _have_methods:
            raise CtpubStop("stop , the grid resume (matrix_%s) of the stored results lacks the strategies needed for the variant comparison (missing %s). The stored grid did not run with the update hierarchy (RUN_A4) on. No new computation was started." % (_f, sorted(_need_methods - _have_methods)))
        _spec = FS_SPEC[_f]
        _banks = list(_spec["banks"])
        _c_ref = CTX[(_f, OUTCOMES[0], "eICU")]
        for _bk in _banks:
            if (_f, _bk) not in _t_ref:
                _t_ref[(_f, _bk)] = ct_time_one_fold(_c_ref, _bk, f"{_f}|{_bk}")
        _R = Resumable(f"ctpub_{_f}")
        _units = [(_f, oc, tg, bk) for oc in OUTCOMES for tg in _spec["targets"] for bk in _banks]
        _todo = [u for u in _units if not _R.unit_done(u)]
        _sec = 0.0
        for u in _todo:
            _t1, _pool_ref = _t_ref[(_f, u[3])]
            _scale = (len(CTX[(u[0], u[1], u[2])]["y"]) / float(len(_c_ref["y"]))) if u[3] == "full" else 1.0
            _sec += (1 if u[3] == "full" else N_DRAWS_MAIN) * N_FOLDS * _t1 * _scale
        _sec_total += _sec
        _plan_rows.append({"item": f"grid {_f} (published procedure)", "unit": len(_units), "units to run": len(_todo), "units to skip": len(_units) - len(_todo),
                           "number of fold fits": sum((1 if u[3] == "full" else N_DRAWS_MAIN) * N_FOLDS for u in _todo), "expected minutes (approximate)": int(round(_sec / 60.0))})
    CT_RUN_HOSP = bool(RUN_HOSPITAL and HAVE_HOSPITALID and EICU_TAB is not None)
    if CT_RUN_HOSP:
        CT_HID, CT_SITES = prepare_sites()
        for _f in [f for f in CT_FS if FS_SPEC[f]["hospital"]]:
            _n_need = sum(len(CT_SITES[oc]) for oc in OUTCOMES)
            _n_have = ct_count_units(f"hospital_{_f}")
            if _n_have != _n_need:
                raise CtpubStop("stop , the number of hospital units of the stored results differs from what these settings require (feature set %s: finished %d, needed %d). This folder (%s) is not the folder of the stored results or its settings differ. "
                                "No new computation was started. Open this notebook in the same folder as the stored results (where out_pram is)." % (_f, _n_have, _n_need, RESUME_DIR.resolve()))
            _RH = Resumable(f"ctpubhosp_{_f}")
            _hu = [(_f, oc, int(h)) for oc in OUTCOMES for h in CT_SITES[oc]]
            _htodo = [u for u in _hu if not _RH.unit_done(u)]
            _t_small = min(v[0] for k, v in _t_ref.items() if k[0] == _f)
            _sec_h = len(_htodo) * N_SITE_SPLITS * _t_small
            _sec_total += _sec_h
            _plan_rows.append({"item": f"hospital level {_f} (published procedure)", "unit": len(_hu), "units to run": len(_htodo), "units to skip": len(_hu) - len(_htodo),
                               "number of fold fits": len(_htodo) * N_SITE_SPLITS, "expected minutes (approximate)": int(round(_sec_h / 60.0))})
    CT_PLAN = pd.DataFrame(_plan_rows)
    emit_table("what this run will compute (counted by timing the actual fit of one fold. The time of the cell with the smallest bank size was used for the hospital folds)", CT_PLAN)
    emit(f"- units to run {int(CT_PLAN['units to run'].sum())} | units to skip {int(CT_PLAN['units to skip'].sum())} | expected about {int(round(_sec_total / 60.0))} min (rough estimate) "
         "(data loading and frozen model fit about 75 seconds and result aggregation are separate). Not one line is written to existing resume files.")
    status("closed-testing pre-check", "done", f"new unit {int(CT_PLAN['units to run'].sum())} columns")


## Grid: run the published procedure in the same cells (resumable)

The same cells (feature set x outcome x target x bank size), the same 5-fold split, the same draw key (`m23`) and the same number of draws (`N_DRAWS_MAIN`, 1 for the full bank) as the grid of notebook 01. For each draw, the 5-fold test predictions are pooled into one bank result for the metrics (the same unit as the variant). Only the published procedure (`ct_pub`) is fitted. The per-fold diagnostics (step, test p, rank, separation) accumulate in `ctpubinfo_*`.


In [ ]:
# ================================================================
# PRAM closed testing  published procedure on the grid (resumable)
# ================================================================
CTP, CTPINFO = {}, {}
if not DATA_OK:
    status("closed-testing grid", "not available", "main inputs absent")
else:
    t_ct_all = time.time()
    for _f in CT_FS:
        _spec = FS_SPEC[_f]
        _R = Resumable(f"ctpub_{_f}")
        _RI = Resumable(f"ctpubinfo_{_f}")
        _units = [(_f, oc, tg, bk) for oc in OUTCOMES for tg in _spec["targets"] for bk in _spec["banks"]]
        emit(f"### grid {_f} published procedure , units {len(_units)} (completed {sum(_R.unit_done(u) for u in _units)} )")
        for u in _units:
            if _R.unit_done(u):
                print("  [SKIP]", u)
                continue
            t0 = time.time()
            _rows, _frows = run_ctpub_unit(CTX[(u[0], u[1], u[2])], u[0], u[1], u[2], u[3], N_DRAWS_MAIN)
            for _r in _rows:
                _R.add_row(_r)
            for _r in _frows:
                _RI.add_row(_r)
            _R.mark_unit(u)
            _RI.mark_unit(u)
            print(f"  [DONE] published procedure {u[1]} / {u[2]} / bank={u[3]}  ({time.time() - t0:.0f}s, {len(_rows)} rows, folds {len(_frows)})", flush=True)
        _dr, _fr = _R.df(), _RI.df()
        _kd = ["fs", "outcome", "target", "bank", "rep", "method"]
        _kf = ["fs", "outcome", "target", "bank", "rep", "fold"]
        CTP[_f] = _dr.drop_duplicates(subset=_kd, keep="last").reset_index(drop=True) if len(_dr) else _dr       # duplicate lines made by resuming after an interruption: only the last one is used
        CTPINFO[_f] = _fr.drop_duplicates(subset=_kf, keep="last").reset_index(drop=True) if len(_fr) else _fr
        save_csv(CTP[_f], f"CTPUB_draws_{_f}")
        save_csv(CTPINFO[_f], f"CTPUB_fold_steps_{_f}")
    status("closed-testing grid", "done", f"{(time.time() - t_ct_all) / 60:.0f} min")


## Hospital level: run the published procedure in the same hospitals and the same folds (resumable)

With the same hospital list and the same 3-fold split (key `a29`) as the hospital-level computation of notebook 01, only the published procedure is fitted. The hospital id is used only inside this notebook, and the result csv contains only pseudonymous codes (`H001` sequence). The bank within a hospital is 2/3 of its stays, so there are few events. **Whether the unpenalized revision exists in that bank (separation) is the key diagnostic of this section.**


In [ ]:
# ================================================================
# PRAM closed testing  published procedure at the hospital level (resumable)
# ================================================================
CTH, CTHINFO = {}, {}
if not (DATA_OK and CT_RUN_HOSP):
    status("closed-testing hospital level", "not available", "main inputs absent or no hospital id column or RUN_HOSPITAL=False")
else:
    t_cth = time.time()
    for _f in [f for f in CT_FS if FS_SPEC[f]["hospital"]]:
        _R, _RF = Resumable(f"ctpubhosp_{_f}"), Resumable(f"ctpubhospfold_{_f}")
        for _oc in OUTCOMES:
            for _i, _h in enumerate(CT_SITES[_oc]):
                _u = (_f, _oc, int(_h))
                if _R.unit_done(_u):
                    continue
                _rows, _frows = ctpub_hospital_unit(CTX[(_f, _oc, "eICU")], _oc, _h, CT_HID[_oc])
                for _r in (_rows or []):
                    _R.add_row(_r)
                for _r in (_frows or []):
                    _RF.add_row(_r)
                _R.mark_unit(_u)
                _RF.mark_unit(_u)
                if _i % 20 == 0:
                    print(f"  [{_f}] {_oc} {_i + 1}/{len(CT_SITES[_oc])} ({(time.time() - t_cth) / 60:.1f} min)", flush=True)
        _dh, _fh = _R.df(), _RF.df()
        CTH[_f] = _dh.drop_duplicates(subset=["outcome", "hospitalid", "method"], keep="last").reset_index(drop=True) if len(_dh) else _dh
        CTHINFO[_f] = _fh.drop_duplicates(subset=["outcome", "hospitalid", "fold"], keep="last").reset_index(drop=True) if len(_fh) else _fh
        for _nm, _df in ((f"CTPUB_hospital_results_{_f}", CTH[_f]), (f"CTPUB_hospital_fold_steps_{_f}", CTHINFO[_f])):
            if len(_df):
                _x = _df.copy()
                _x.insert(1, "site", _x["hospitalid"].map(SITE_CODE))
                save_csv(_x.drop(columns=["hospitalid"]), _nm)
    status("closed-testing hospital level", "done", f"{len(CTH)} feature sets {(time.time() - t_cth) / 60:.0f} min")


## Report: comparison, step distribution, side by side with the variant, comparison with PRAM

Three things are produced at once.

1. **Pairing check**: whether the two were made from the same bank is verified with the stored step records of the variant. `n_bank` and `events_bank` must be equal per fold, and the variant step recomputed by this notebook (`step_var_ridge`) must equal `ct_step_b1` stored by notebook 01. If any differ, the status becomes comparison mismatch and the value must not be used.
2. **Tables with the same definition as Supplementary S7**: `CTPUB_S7A_table` (AUROC change versus the frozen logistic and step fractions), `CTPUB_S7B_table` (ECE and net benefit), `CTPUB_S7C_flags` (number of cells where calibration and net benefit are equal or better than retrieval without losing discrimination), `CTPUB_vs_pram` (pair difference of the published procedure versus retrieval). The values of the variant are read from the resume of notebook 01.
3. **Hospital level**: the same columns as Supplementary S8 Panel B (`CTPUB_hospital_summary`). The same summary function is run on the stored hospital rows and first checked against `hospital_summary.csv` of notebook 01.

Why the steps differ is separated by four columns of `CTPUB_step_shares` (`share_diff_order` test order, `share_diff_fit_var` and `share_diff_fit_pub` revision fit, `share_diff_pub_vs_var` their sum).


In [ ]:
# ================================================================
# PRAM closed testing  report
# ================================================================
CT_CHECK, CT_STEPS, CT_S7A, CT_S7B, CT_S7C, CT_VSPRAM, CT_HSUM, CT_HCHECK, CT_SIDE = (pd.DataFrame() for _ in range(9))
CT_CHECK_BAD = 0
CT_METHODS_S7 = ["b1_base", "upd1_b1", "upd2_b1", "upd3_b1", "anchored_ridge", "ct_b1", "ct_pub", "pram_b1", "platt_b1"]


def ct_hosp_summary(df, fs):
    """hospital-level summary. Columns with the same definition as hosp_summary_table (hospital summary cell of the stored results) (used in Supplementary S8 Panel B)."""
    out = []
    for (oc, m), s in df.groupby(["outcome", "method"], sort=False):
        d = s["d_auroc_b1"].to_numpy(float)
        w = s["n_site"].to_numpy(float)
        out.append({"fs": fs, "outcome": oc, "method": m, "k": int(len(s)),
                    "delta_stays_weighted": float(np.average(d, weights=w)), "delta_median": float(np.median(d)),
                    "delta_p10": float(np.quantile(d, 0.10)), "delta_p90": float(np.quantile(d, 0.90)),
                    "pct_better": float(100 * (d > 0).mean()),
                    "slope_median": float(s["slope"].median()), "ece_median": float(s["ece"].median()),
                    "nb_20_median": float(s[nbcol(THRESH_MS)].median()),
                    "nb_site_median": float(np.median([r[nbcol(T_SITE[oc])] for r in s.to_dict("records")]))})
    return pd.DataFrame(out)


def ct_share_table(fold_df, group_cols):
    """step fractions per fold | disagreement fraction of the two procedures | cause of disagreement (test order, revision fit) | diagnostics (separation, rank deficiency, non-convergence)."""
    g = fold_df.groupby(group_cols, sort=False)
    rows = []
    for key, s in g:
        key = key if isinstance(key, tuple) else (key,)
        row = dict(zip(group_cols, key))
        row["n_folds"] = int(len(s))
        for k, nm in CT_STEP_LABEL.items():
            row[f"pub_share_{k}_{nm}"] = float((s["step_pub"] == k).mean())
        for k, nm in CT_STEP_LABEL.items():
            row[f"var_share_{k}_{nm}"] = float((s["step_var_stored"] == k).mean()) if "step_var_stored" in s else float("nan")
        row["share_diff_pub_vs_var"] = float((s["step_pub"] != s["step_var_stored"]).mean()) if "step_var_stored" in s else float("nan")
        if "rep" in s and "step_var_stored" in s:               # per draw: a draw differs if the step differs in any one of the 5 folds
            row["share_draws_any_diff"] = float((s["step_pub"] != s["step_var_stored"]).groupby(s["rep"]).any().mean())
        else:
            row["share_draws_any_diff"] = float("nan")
        row["share_diff_order"] = float((s["step_pub_ridge"] != s["step_var_ridge"]).mean())
        row["share_diff_fit_var"] = float((s["step_var_exact"] != s["step_var_ridge"]).mean())
        row["share_diff_fit_pub"] = float((s["step_pub"] != s["step_pub_ridge"]).mean())
        okm = s["ok"].astype(bool)
        row["n_ok"] = int(okm.sum())
        so = s[okm]
        row["share_separated"] = float(so["separated"].astype(bool).mean()) if len(so) else float("nan")
        row["share_rank_deficient"] = float((so["rank"] < so["d"] + 1).mean()) if len(so) else float("nan")
        row["share_not_converged"] = float((~so["converged"].astype(bool)).mean()) if len(so) else float("nan")
        rows.append(row)
    return pd.DataFrame(rows)


if not (DATA_OK and CTP):
    status("closed-testing report", "not available", "no grid results")
else:
    _chk_rows, _step_parts, _s7a, _s7b, _s7c, _vsp = [], [], [], [], [], []
    for _f in CT_FS:
        _stored = Resumable(f"matrix_{_f}").df()
        _sinfo = Resumable(f"matrixinfo_{_f}").df()
        _fold = CTPINFO[_f].merge(_sinfo[["outcome", "target", "bank", "rep", "fold", "n_bank", "events_bank", "ct_step_b1", "ct_p_all_b1", "ct_p_rev_vs_recal_b1"]],
                                  on=["outcome", "target", "bank", "rep", "fold"], how="left", suffixes=("", "_stored"))
        _fold["step_var_stored"] = _fold["ct_step_b1"]
        _n_missing = int(_fold["ct_step_b1"].isna().sum())
        _bad_bank = int(((_fold["n_bank"] != _fold["n_bank_stored"]) | (_fold["events_bank"] != _fold["events_bank_stored"])).sum())
        _bad_step = int((_fold["step_var_ridge"] != _fold["step_var_stored"]).sum())
        _chk_rows.append({"scope": f"grid {_f}", "number of folds": int(len(_fold)), "no stored record": _n_missing, "bank size or event count mismatch": _bad_bank, "variant step mismatch": _bad_step})
        _step_parts.append(_fold)
        _all = pd.concat([_stored[_stored["method"].isin(CT_METHODS_S7)], CTP[_f]], ignore_index=True)
        _g = draw_median_table(_all, CT_METHODS_S7)
        _gp = pd.concat([paired_gap(_all, m, "b1_base") for m in CT_METHODS_S7 if m != "b1_base"], ignore_index=True)
        _g = _g.merge(_gp[["fs", "outcome", "target", "bank", "method", "gap_median", "share_below_zero"]], on=["fs", "outcome", "target", "bank", "method"], how="left")
        _g["o"] = [bank_sort_key(b) for b in _g["bank"]]
        _g = _g.sort_values(["fs", "outcome", "target", "o", "method"]).drop(columns=["o"])
        _s7a.append(_g.rename(columns={"gap_median": "dAUROC_vs_b1"}))
        _w = _g.pivot_table(index=["fs", "outcome", "target", "bank"], columns="method", values=["auroc", "ece", "nb_site"], aggfunc="first")
        _w.columns = [f"{a}_{b}" for a, b in _w.columns]
        _w = _w.reset_index()
        for _tag in ("ct_b1", "ct_pub"):
            _w[f"flag_{_tag}"] = ((_w[f"nb_site_{_tag}"] >= _w["nb_site_pram_b1"] - 0.0005) & (_w[f"ece_{_tag}"] < _w["ece_pram_b1"])
                                  & (_w[f"auroc_{_tag}"] >= _w["auroc_b1_base"] - 0.002))
        _s7b.append(_w)
        _vp = paired_gap(_all, "ct_pub", "pram_b1")
        if len(_vp):
            _vp = _vp.rename(columns={"method": "method_a", "base": "method_b"})
        _vsp.append(_vp)
        _vv = paired_gap(_all, "ct_b1", "pram_b1")
        if len(_vv):
            _vsp.append(_vv.rename(columns={"method": "method_a", "base": "method_b"}))
    _foldall = pd.concat(_step_parts, ignore_index=True)
    CT_CHECK = pd.DataFrame(_chk_rows)
    CT_STEPS = ct_share_table(_foldall, ["fs", "outcome", "target", "bank"])
    CT_STEPS["o"] = [bank_sort_key(b) for b in CT_STEPS["bank"]]
    CT_STEPS = CT_STEPS.sort_values(["fs", "outcome", "target", "o"]).drop(columns=["o"]).reset_index(drop=True)
    CT_S7A = pd.concat(_s7a, ignore_index=True)
    CT_S7B = pd.concat(_s7b, ignore_index=True)
    CT_S7C = CT_S7B.groupby(["fs", "bank"], sort=False).agg(n_cells=("flag_ct_pub", "size"), n_flag_variant=("flag_ct_b1", "sum"), n_flag_published=("flag_ct_pub", "sum")).reset_index()
    CT_S7C["o"] = [bank_sort_key(b) for b in CT_S7C["bank"]]
    CT_S7C = CT_S7C.sort_values(["fs", "o"]).drop(columns=["o"])
    CT_VSPRAM = pd.concat([v for v in _vsp if len(v)], ignore_index=True)
    _wd = CT_S7A.pivot_table(index=["fs", "outcome", "target", "bank"], columns="method", values="dAUROC_vs_b1", aggfunc="first")
    _wd = _wd[[m for m in ("upd1_b1", "upd2_b1", "upd3_b1", "ct_b1", "ct_pub", "anchored_ridge", "pram_b1") if m in _wd.columns]].add_prefix("dAUROC_").reset_index()
    _sc = ["fs", "outcome", "target", "bank"] + [f"var_share_{k}_{nm}" for k, nm in CT_STEP_LABEL.items()] + [f"pub_share_{k}_{nm}" for k, nm in CT_STEP_LABEL.items()] + ["share_diff_pub_vs_var", "share_draws_any_diff", "share_separated"]
    CT_SIDE = _wd.merge(CT_STEPS[_sc], on=["fs", "outcome", "target", "bank"], how="left")
    CT_SIDE["o"] = [bank_sort_key(b) for b in CT_SIDE["bank"]]
    CT_SIDE = CT_SIDE.sort_values(["fs", "outcome", "target", "o"]).drop(columns=["o"]).reset_index(drop=True)
    CT_S7B_OUT = CT_S7B[["fs", "outcome", "target", "bank"] + [c for c in CT_S7B.columns if c.split("_")[0] in ("ece", "nb") and c.endswith(("b1_base", "ct_b1", "ct_pub", "pram_b1", "platt_b1", "upd2_b1"))]
                        + ["flag_ct_b1", "flag_ct_pub"]]
    CT_CHECK_BAD = int(CT_CHECK[["no stored record", "bank size or event count mismatch", "variant step mismatch"]].to_numpy().sum())
    emit_table("pairing check (grid): were they made from the same bank. All three columns must be 0", CT_CHECK)
    emit(f"- grid mismatch total **{CT_CHECK_BAD}** , " + ("if 0, the bank size and event count are the same per fold, and the variant steps recomputed by this notebook equal the stored values of the stored results." if CT_CHECK_BAD == 0
                                                  else "is not 0. This value must not be used. For the mismatched folds compare step_var_ridge and step_var_stored in CTPUB_fold_steps_*."))
    for _f in CT_FS:
        _x = CT_STEPS[CT_STEPS["fs"] == _f]
        _cols = ["outcome", "target", "bank", "n_folds"] + [c for c in _x.columns if c.startswith(("pub_share_", "var_share_"))] + ["share_diff_pub_vs_var", "share_draws_any_diff", "share_diff_order", "share_diff_fit_var", "share_diff_fit_pub"]
        emit_table(f"step fractions , {_f} (per fold x draw. pub = published procedure, var = variant)", _x[_cols], digits=3, max_rows=60)
        emit_table(f"diagnostics , {_f} (fraction of folds with separation, rank deficiency or non-convergence of the unpenalized revision)",
                   _x[["outcome", "target", "bank", "n_folds", "share_separated", "share_rank_deficient", "share_not_converged"]], digits=3, max_rows=60)
    _a_cols = ["fs", "outcome", "target", "bank", "method", "n_draws", "auroc", "dAUROC_vs_b1", "ece", "nb_site", "nb_20"]
    emit_table("table to put in Supplementary S7 Panel A (published procedure and variant on one line: AUROC change is the median over draws versus the frozen logistic, step fractions are per fold, var is the variant and pub is the published procedure)", CT_SIDE, digits=4, max_rows=60)
    emit_table("same definition as Supplementary S7 Panel A (median over draws of the AUROC change versus the frozen logistic, ct_pub is the published procedure, ct_b1 is the variant)",
               CT_S7A[CT_S7A["method"].isin(["b1_base", "upd2_b1", "ct_b1", "ct_pub", "pram_b1"])][_a_cols], max_rows=400)
    emit_table("number of decision cells (same rule as Supplementary S7 Panel C: ECE lower than retrieval, net benefit not lower by more than 0.0005, AUROC not lower than the frozen model by more than 0.002)", CT_S7C, digits=3)
    emit_table("published procedure versus retrieval, variant versus retrieval (AUROC pair difference between the same draws)", CT_VSPRAM, max_rows=200)
    save_csv(CT_CHECK, "CTPUB_check_vs_stored")
    save_csv(CT_S7A, "CTPUB_S7A_table")
    save_csv(CT_SIDE, "CTPUB_S7A_side_by_side")
    save_csv(CT_S7B_OUT, "CTPUB_S7B_table")
    save_csv(CT_S7C, "CTPUB_S7C_flags")
    save_csv(CT_VSPRAM, "CTPUB_vs_pram")
    # ---- hospital level ----
    if CTH:
        _hs, _hc = [], []
        for _f in CTH:
            _hst = Resumable(f"hospital_{_f}").df()
            _hsi = Resumable(f"hospital_folds_{_f}").df()
            _hsum_old = ct_hosp_summary(_hst[_hst["method"].isin(["b1_base", "ct_b1", "upd1_b1", "upd2_b1", "upd3_b1", "anchored_ridge", "pram_b1"])], _f)
            _new = CTH[_f].copy()
            _hsum_new = ct_hosp_summary(_new, _f)
            _hs.append(pd.concat([_hsum_old, _hsum_new], ignore_index=True))
            _hf = CTHINFO[_f].merge(_hsi[["outcome", "hospitalid", "fold", "n_bank", "events_bank", "ct_step_b1"]], on=["outcome", "hospitalid", "fold"], how="left", suffixes=("", "_stored"))
            _hf["step_var_stored"] = _hf["ct_step_b1"]
            _hc.append({"scope": f"hospital {_f}", "number of folds": int(len(_hf)), "no stored record": int(_hf["ct_step_b1"].isna().sum()),
                        "bank size or event count mismatch": int(((_hf["n_bank"] != _hf["n_bank_stored"]) | (_hf["events_bank"] != _hf["events_bank_stored"])).sum()),
                        "variant step mismatch": int((_hf["step_var_ridge"] != _hf["step_var_stored"]).sum())})
            _hs_steps = ct_share_table(_hf.assign(bank="hospital", target="eICU", fs=_f), ["fs", "outcome", "target", "bank"])
            CT_STEPS = pd.concat([CT_STEPS, _hs_steps], ignore_index=True)
        CT_HSUM = pd.concat(_hs, ignore_index=True)
        CT_HCHECK = pd.DataFrame(_hc)
        _hb = int(CT_HCHECK[["no stored record", "bank size or event count mismatch", "variant step mismatch"]].to_numpy().sum())
        CT_CHECK_BAD += _hb
        # is the summary made from the stored hospital rows equal to hospital_summary.csv of the stored results
        _csv3 = CSV_DIR / "hospital_summary.csv"
        _cmp_note = f"the stored `hospital_summary.csv` does not exist, so no comparison was made (path searched `{_csv3.resolve()}`)"
        _cmp_bad = None
        if _csv3.is_file():
            _old3 = pd.read_csv(_csv3)
            _mine = CT_HSUM[CT_HSUM["method"] != "ct_pub"]
            _j = _mine.merge(_old3, on=["fs", "outcome", "method"], suffixes=("", "_csv3"))
            _vc = ["k", "delta_stays_weighted", "delta_median", "delta_p10", "delta_p90", "pct_better", "slope_median", "ece_median", "nb_20_median", "nb_site_median"]
            _cmp_bad = int(sum((~np.isclose(_j[c].astype(float), _j[c + "_csv3"].astype(float), rtol=1e-4, atol=1e-6)).sum() for c in _vc)) if len(_j) else -1
            _cmp_note = f"rows compared with the stored `hospital_summary.csv` {len(_j)} x columns {len(_vc)} , mismatches {_cmp_bad} (csv has 6 significant digits, so a relative 1e-4 is allowed)"
            CT_CHECK_BAD += max(_cmp_bad, 0)
        emit_table("pairing check (hospital): is the bank the same per fold, are the variant steps equal to the stored values", CT_HCHECK)
        emit(f"- hospital mismatch total **{_hb}** · {_cmp_note}")
        _hshow = CT_HSUM[CT_HSUM["method"].isin(["b1_base", "upd2_b1", "ct_b1", "ct_pub", "pram_b1"])]
        emit_table("same columns as Supplementary S8 Panel B (hospital level, versus frozen logistic. ct_pub is the published procedure, ct_b1 is the variant)", _hshow, max_rows=60)
        CT_STEPS = CT_STEPS.reset_index(drop=True)
        emit_table("hospital-level step fractions and diagnostics (per fold)", CT_STEPS[CT_STEPS["bank"] == "hospital"][["fs", "outcome", "n_folds"] + [c for c in CT_STEPS.columns if c.startswith(("pub_share_", "var_share_"))]
                                                                              + ["share_diff_pub_vs_var", "share_diff_order", "share_diff_fit_var", "share_diff_fit_pub", "share_separated", "share_rank_deficient", "share_not_converged"]], digits=3, max_rows=20)
        save_csv(CT_HSUM, "CTPUB_hospital_summary")
        save_csv(CT_HCHECK, "CTPUB_hospital_check_vs_stored")
    save_csv(CT_STEPS, "CTPUB_step_shares")
    status("closed-testing report", "done" if CT_CHECK_BAD == 0 else "comparison mismatch", f"pairing mismatch total {CT_CHECK_BAD}")


## Closing: summary, map of result files, completion check, result zip

This cell does three things: (1) writes the key numbers and all tables shown on the screen to `PRAM-closed-testing-summary.md` and `PRAM-closed-testing-results.md` (the summary and results md of notebook 01 are not touched); (2) gives the map of which csv goes to which table position; (3) makes a zip holding only the `CTPUB_` csv files and the closed-testing md (if there is a patient or hospital identifier column, it does not make the zip and stops; the resume folder and the files of notebook 01 are not included).

**The zip is created in the current working folder.**


In [ ]:
# ================================================================
# PRAM closed testing  closing . summary | map of outputs | was it finished | zip
# ================================================================
CT_MAP = pd.DataFrame([
    ["Closed testing column and step fraction columns of Supplementary S7 Panel A", "CTPUB_S7A_side_by_side (dAUROC_ct_pub and pub_share_* are the published procedure, dAUROC_ct_b1 and var_share_* are the variant) | the long table of the same values is CTPUB_S7A_table", "the two are side by side on one line"],
    ["ECE and net benefit of Supplementary S7 Panel B", "CTPUB_S7B_table (ece_ct_pub · nb_site_ct_pub)", ""],
    ["number of decision cells of Supplementary S7 Panel C", "CTPUB_S7C_flags (n_flag_published, n_flag_variant for the variant)", ""],
    ["Supplementary S7 note (difference between the variant and the published procedure)", "CTPUB_step_shares (share_diff_pub_vs_var · share_diff_order · share_diff_fit_*)", "where the statement that there are two differences is closed by values"],
    ["closed testing in Results and the Discussion", "CTPUB_S7A_table · CTPUB_vs_pram", "the closed-testing values of the published procedure next to the variant"],
    ["Closed testing row of Supplementary S8 Panel B (hospital level)", "CTPUB_hospital_summary (method = ct_pub)", ""],
    ["pairing evidence", "CTPUB_check_vs_stored · CTPUB_hospital_check_vs_stored", "valid only if the mismatch is 0"],
], columns=["place in the paper", "csv (github_csv folder)", "note"])


def ct_headline_lines():
    out = []
    if len(CT_STEPS):
        for _f in CT_FS:
            for oc in OUTCOMES:
                for tg in FS_SPEC[_f]["targets"]:
                    s = CT_STEPS[(CT_STEPS["fs"] == _f) & (CT_STEPS["outcome"] == oc) & (CT_STEPS["target"] == tg) & (CT_STEPS["bank"] != "hospital")]
                    if len(s):
                        out.append(f"- step {_f} {oc} {tg}: fraction of folds that chose the revision (published procedure / variant) " + " · ".join(
                            f"{r.bank} {r.pub_share_3_revision:.0%}/{r.var_share_3_revision:.0%}" for r in s.itertuples()) + " , folds in which the two procedures chose different steps " + " · ".join(
                            f"{r.bank} {r.share_diff_pub_vs_var:.0%}" for r in s.itertuples()))
    if len(CT_S7A):
        for _f in CT_FS:
            for oc in OUTCOMES:
                s = CT_S7A[(CT_S7A["fs"] == _f) & (CT_S7A["outcome"] == oc) & (CT_S7A["target"] == "eICU") & CT_S7A["method"].isin(["ct_pub", "ct_b1", "pram_b1"])]
                if len(s):
                    for m in ("ct_pub", "ct_b1", "pram_b1"):
                        t = s[s["method"] == m]
                        out.append(f"- AUROC change (versus frozen logistic) {_f} {oc} eICU {m}: " + " · ".join(f"{r.bank} {r.dAUROC_vs_b1:+.4f}" for r in t.itertuples()))
    if len(CT_S7C):
        for r in CT_S7C.itertuples():
            out.append(f"- decision cells {r.fs} bank {r.bank}: published procedure {int(r.n_flag_published)}/{int(r.n_cells)} | variant {int(r.n_flag_variant)}/{int(r.n_cells)}")
    if len(CT_HSUM):
        for r in CT_HSUM[CT_HSUM["method"].isin(["ct_pub", "ct_b1"])].itertuples():
            out.append(f"- hospital level {r.fs} {r.outcome} {r.method}: mean AUROC change {r.delta_stays_weighted:+.4f}, hospitals that gained {r.pct_better:.1f}%, median ECE {r.ece_median:.4f}, median net benefit {r.nb_site_median:.4f}")
    return out


_HEAD4 = [f"# PRAM closed testing summary (published-procedure closed testing, run {time.strftime('%Y-%m-%d %H:%M')})", "",
          "**The numbers in this file are notebook run results.**", "",
          f"**pairing mismatch total {CT_CHECK_BAD}** (valid only if 0).", "",
          "## Key numbers", ""] + (ct_headline_lines() or ["- (no section was computed)"]) + ["", "## Status by section", "",
          "| section | status | note |", "|---|---|---|"] + [f"| {k} | {v[0]} | {v[1]} |" for k, v in STATUS.items() if k.startswith("closed-testing")] + ["", "## Map to manuscript table positions", "", md_table(CT_MAP), ""]
save_csv(CT_MAP, "CTPUB_manuscript_table_map")
(OUT_DIR / CT_SUM_NAME).write_text("\n".join(_HEAD4), encoding="utf-8")
(OUT_DIR / CT_RES_NAME).write_text("\n".join(REPORT_LINES), encoding="utf-8")
emit_table("status by section (was it finished)", pd.DataFrame([{"section": k, "status": v[0], "note": v[1]} for k, v in STATUS.items() if k.startswith("closed-testing")]), max_rows=20)
_not4 = [k for k, v in STATUS.items() if k.startswith("closed-testing") and v[0] not in ("done", "off", "skipped")]
emit(f"- closed-testing sections not completed {len(_not4)} columns: {_not4 if _not4 else 'absent'}")

# ---- zip: only the CTPUB_ csv files and the closed-testing md. Stops if there is a patient or hospital identifier column ----
ID_LIKE_CT = {"stay_id", "patientunitstayid", "subject_id", "hadm_id", "icustay_id", "hospitalid", "uniquepid", "patienthealthsystemstayid", "pt_idx", "patient_id"}
_bad_ct = []
for _p in sorted(CSV_DIR.glob(CT_CSV_PREFIX + "*.csv")):
    _h = pd.read_csv(_p, nrows=2)
    if ID_LIKE_CT & {c.lower() for c in _h.columns}:
        _bad_ct.append((_p.name, "identifier column"))
if _bad_ct:
    emit(f"**The zip was not made.** There are files with identifier columns: {_bad_ct}")
    status("closed-testing zip", "not available", str(_bad_ct))
else:
    with zipfile.ZipFile(CT_ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as _z:
        for _p in sorted(CSV_DIR.glob(CT_CSV_PREFIX + "*.csv")) + [OUT_DIR / CT_SUM_NAME, OUT_DIR / CT_RES_NAME]:
            _z.write(_p, arcname=_p.relative_to(OUT_DIR.parent).as_posix())
    with zipfile.ZipFile(CT_ZIP_NAME) as _z:
        _names_ct = _z.namelist()
    emit(f"- zip `{Path(CT_ZIP_NAME).resolve()}` ({Path(CT_ZIP_NAME).stat().st_size / 1e6:.2f} MB, files {len(_names_ct)} files, only the CTPUB_ csv files and the closed-testing md)")
    emit("**Keep this zip with the results.**")
    status("closed-testing zip", "done", f"{len(_names_ct)} files")
